# Modelo de riesgo de mora en leasing de motos — Cuaderno único

**Modelos y Simulación de Sistemas II · Universidad de Antioquia · Grupo 14**
Juan Esteban Cardozo Rivera · María Carolina Cardona Calderón · Oliver Andrés Quiroz Mora

## Cómo usarlo (3 pasos)

1. Verifique que la carpeta **`ProyectoModelos2_Colab`** esté en su *Mi unidad* (si se la compartieron: *Compartidos conmigo → clic derecho → Organizar → Agregar acceso directo → Mi unidad*).
2. Menú **Entorno de ejecución → Ejecutar todo** y acepte el permiso de Drive.
3. Espere. La **celda de verificación** (justo abajo) le dice si falta algo; al final hay un **control de calidad** que compara los resultados con las cifras del informe.

**Tiempo estimado: 15–30 minutos** (lo más lento es la rejilla K/H/X de la Parte 3). Si se desconecta, vuelva a ejecutar todo.

## Qué contiene

| Parte | Qué responde |
|---|---|
| 1 · EDA | ¿Qué datos hay, cómo se define el episodio y la etiqueta de riesgo grave? |
| 2 · Clustering | ¿Qué perfiles de pago existen? (sin usar la etiqueta) |
| 3 · Clasificación | ¿Qué tan bien se predice el riesgo grave con validación temporal? |
| 4 · Selección y explicabilidad | ¿Qué variables importan y por qué (filtro, Lasso, PCA, SHAP)? |
| 5 · Mejoras y validación externa | ¿Se puede mejorar? ¿Funciona en Sincelejo y Montería? |

Los datos están **anonimizados** (contratos como C00001…). Las celdas con título *⚙️* son de preparación: no hace falta abrirlas.

In [ ]:
#@title ⚙️ Paso 0 · Preparar el entorno y verificar archivos (ejecutar primero)
import importlib, os, subprocess, sys
from pathlib import Path

NOMBRE_CARPETA = "ProyectoModelos2_Colab"
ARCHIVOS = {
    "Código (.py)": ["ciclos.py", "variables.py", "particion.py", "modelado.py", "clasificacion.py"],
    "Datos (.csv.gz)": ["abonos.csv.gz", "motos_diario.csv.gz", "rentabilidad.csv.gz", "dataset_modelo.csv.gz"],
    "Resultados de la Parte 5": ["reports/mejora_trayectoria.csv", "reports/ajuste_hiperparametros.csv",
                                  "reports/mejora_ensamble.csv", "reports/mejora_gestion.csv",
                                  "reports/validacion_externa_sedes.csv"],
}

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    raices = [Path("/content/drive/MyDrive"), *Path("/content/drive").glob(".shortcut-targets-by-id/*")]
    carpeta = None
    for patron in (NOMBRE_CARPETA, f"*/{NOMBRE_CARPETA}", f"*/*/{NOMBRE_CARPETA}"):
        for raiz in raices:
            for c in raiz.glob(patron):
                if (c / "clasificacion.py").exists():
                    carpeta = c
                    break
            if carpeta:
                break
        if carpeta:
            break
    if carpeta is None:
        raise FileNotFoundError(
            f"No se encontró la carpeta {NOMBRE_CARPETA} en su Drive. Si se la compartieron: Google Drive > "
            "Compartidos conmigo > clic derecho sobre la carpeta > Organizar > Agregar acceso directo > Mi unidad. "
            "Luego Entorno de ejecución > Reiniciar y ejecutar todo.")
    os.chdir(carpeta)
    print("Carpeta de trabajo:", carpeta)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imbalanced-learn", "shap", "umap-learn"], check=False)
else:
    print("Ejecutando fuera de Colab, carpeta:", Path.cwd())

print("\nVerificación de archivos")
faltan = []
for grupo, lista in ARCHIVOS.items():
    for a in lista:
        ok = Path(a).exists()
        print(("  ✅ " if ok else "  ❌ ") + f"{a}  ({grupo})")
        if not ok:
            faltan.append(a)
print("\nVerificación de librerías")
for lib in ("pandas", "numpy", "sklearn", "scipy", "matplotlib", "imblearn", "shap", "umap"):
    try:
        m = importlib.import_module(lib)
        print(f"  ✅ {lib} {getattr(m, '__version__', '')}")
    except Exception as e:
        print(f"  ❌ {lib}: {e}")
        faltan.append(lib)
if faltan:
    raise FileNotFoundError("Falta lo siguiente (súbalo a la carpeta de Drive o instale la librería): " + ", ".join(faltan))
print("\n✅ Todo listo. Continúe con la Parte 1 (o ejecute todo).")

# PARTE 1 · Exploración de datos (EDA)

<sub>Contenido del cuaderno `01_eda.ipynb`.</sub>

# Entregable I — Análisis exploratorio de datos

**Modelo predictivo de riesgo de mora en contratos de leasing de motocicletas**
Modelos y Simulación de Sistemas II — Universidad de Antioquia — Grupo 14

Este notebook caracteriza la base de datos real, define la variable objetivo con su
corte temporal y construye las variables candidatas de las primeras *k* semanas de
cada contrato. Sus resultados alimentan las secciones 1.5 a 1.8 del documento.

**Datos.** Trabaja sobre la copia anonimizada en `data/anon/`, generada con
`src/extraer.py` (ver `README.md`). Los datos son confidenciales de la empresa y no
están en el repositorio. Las reglas de negocio aplicadas están en
`docs/reglas_de_negocio.md`; la lógica del calendario y de la etiqueta está en
`src/ciclos.py` y `src/variables.py`, con pruebas en `tests/`.

**Alcance.** Sede Cartagena. Los montos están en miles de pesos.

**En Google Colab.** Abra este notebook desde la carpeta compartida
`ProyectoModelos2_Colab` (si aparece en "Compartidos conmigo", agréguele antes un acceso
directo en "Mi unidad") y ejecute todo. La primera celda de código conecta Drive y
entra a la carpeta sola; los resultados se guardan en la sesión (`/content/salidas`),
no en la carpeta compartida.

In [ ]:
#@title ⚙️ Preparación de esta parte (no hace falta tocar)
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Funciona con la estructura del repositorio (src/, data/anon/) o con todos los
# archivos sueltos en una misma carpeta, como quedan en la carpeta compartida de Drive.
EN_COLAB = "google.colab" in sys.modules
NOMBRE_CARPETA = "ProyectoModelos2_Colab"


# Ubica la carpeta compartida en Drive: en Mi unidad, dentro de otra carpeta o
# detras de un acceso directo (Colab los expone en .shortcut-targets-by-id).
def buscar_carpeta_drive():
    raices = [Path("/content/drive/MyDrive"), *Path("/content/drive").glob(".shortcut-targets-by-id/*")]
    for patron in (NOMBRE_CARPETA, f"*/{NOMBRE_CARPETA}", f"*/*/{NOMBRE_CARPETA}"):
        for raiz in raices:
            for carpeta in raiz.glob(patron):
                if (carpeta / "ciclos.py").exists():
                    return carpeta
    return None


if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    carpeta = buscar_carpeta_drive()
    if carpeta is None:
        raise FileNotFoundError(
            f"No se encontro la carpeta {NOMBRE_CARPETA} en su Drive. Si se la compartieron, "
            "vaya a Google Drive > Compartidos conmigo, clic derecho sobre la carpeta > "
            "Organizar > Agregar acceso directo > Mi unidad, y vuelva a ejecutar todo "
            "(Entorno de ejecucion > Reiniciar y ejecutar todo).")
    os.chdir(carpeta)
    print("Carpeta de trabajo:", carpeta)
RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
# En Colab los resultados van a la sesion y no a la carpeta compartida: quien la
# tiene en solo lectura puede ejecutar igual y nadie sobrescribe archivos de otros.
DIR_SALIDA = Path("/content/salidas") if EN_COLAB else RAIZ


def ubicar(archivos, carpetas):
    for carpeta in carpetas:
        if all((carpeta / a).exists() for a in archivos):
            return carpeta
    raise FileNotFoundError(f"No se encontraron {archivos} en {[str(c) for c in carpetas]}. "
                            "Suba esos archivos (ver README) a la misma carpeta del notebook.")


DIR_SRC = ubicar(["ciclos.py", "variables.py"], [RAIZ / "src", Path.cwd()])
DIR_DATOS = ubicar(["abonos.csv.gz", "motos_diario.csv.gz", "rentabilidad.csv.gz"], [RAIZ / "data" / "anon", Path.cwd()])
sys.path.insert(0, str(DIR_SRC))
from ciclos import construir_episodios, pagos_por_dia, construir_ciclos, retiros, CARTAS_RETIRO
from variables import asignar_retiro, marcar_poblacion, etiqueta, variables_entrada

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
DIR_FIG = DIR_SALIDA / "reports" / "figuras"
DIR_FIG.mkdir(parents=True, exist_ok=True)

# Paleta validada (guia de visualizacion): azul y naranja son las dos primeras
# posiciones categoricas; el texto usa tinta neutra, nunca el color de la serie.
AZUL, NARANJA, TINTA, TINTA_2, REJILLA = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#e6e5e0"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 10,
    "axes.edgecolor": TINTA_2, "axes.labelcolor": TINTA, "xtick.color": TINTA_2, "ytick.color": TINTA_2,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": REJILLA,
    "axes.axisbelow": True, "axes.titlesize": 11, "axes.titleweight": "bold",
})
CIFRAS = {}  # cifras clave que se exportan para redactar el documento

## 1. Parámetros del estudio

Todos los resultados dependen de estos cuatro valores. Se justifican con evidencia en
la sección 7; cambiarlos aquí recalcula todo el notebook.

| Parámetro | Valor | Significado |
|---|---|---|
| `K` | 8 semanas | ventana de observación: información disponible para predecir |
| `H` | 12 semanas | horizonte de predicción, inmediatamente después de la ventana |
| `X` | 14 días | atraso de un ciclo que define "riesgo grave de mora" (dos cuotas semanales) |
| `INICIO_COHORTE` | 2024-03-30 | primer día de la foto diaria; antes solo se ven los episodios que sobrevivieron |

In [ ]:
K, H, X = 8, 12, 14
INICIO_COHORTE = pd.Timestamp("2024-03-30")

## 2. Carga de datos

In [ ]:
abonos = pd.read_csv(DIR_DATOS / "abonos.csv.gz", parse_dates=["fecha"])
diario = pd.read_csv(DIR_DATOS / "motos_diario.csv.gz", parse_dates=["fecha_registro"], dtype={"justificacion": str})
rentab = pd.read_csv(DIR_DATOS / "rentabilidad.csv.gz", parse_dates=["fecha_inicio", "fecha_final", "fecha_registro"])
CORTE = abonos.fecha.max()
print(f"Corte de datos: {CORTE.date()}")
for nombre, df in (("abonos", abonos), ("motos_diario", diario), ("rentabilidad", rentab)):
    print(f"{nombre:13s} {len(df):>9,d} filas  {df.shape[1]:>2d} columnas  {df.id_contrato.nunique():>5,d} contratos")

## 3. Diccionario de datos

La copia ya excluye, desde la extracción, los registros fuera de alcance (empresa
RASTREOYA, identificadores con correo o no numéricos y las ventas a Medellín) y todas
las columnas con datos personales. El contrato se reemplazó por un código (`id_contrato`).

In [ ]:
SIGNIFICADO = {
    "abonos": {
        "id_contrato": "Código anónimo del contrato (identifica la moto; se reasigna entre clientes)",
        "fecha": "Fecha del registro de pago",
        "abono": "Pago al canon (reduce el saldo de la moto)",
        "transf": "Pago al canon por transferencia (reduce el saldo; poco común)",
        "multa": "Multa por atraso cobrada en el pago (no reduce el saldo)",
        "repuesto": "Devolución de dinero prestado (repuestos, trámites, SOAT); no reduce el saldo",
        "carta": "SI = abono ficticio que cierra el saldo (la moto deja de estar con el cliente)",
        "cartamotivo": "Motivo de la carta (regreso diario, administración, realquiler, pérdida, pronto pago...)",
        "concepto": "Canal o tipo de registro (EFECTIVO, DIGITACION, EGRESOS...)",
        "empresa": "Empresa del grupo (MOTOHOY, RENTAYA)",
    },
    "motos_diario": {
        "id_contrato": "Código anónimo del contrato",
        "fecha_registro": "Día de la foto diaria",
        "fecha_inicio": "Inicio del episodio (cliente con la moto); 0000-00-00 = trámites pendientes",
        "tipopago": "Frecuencia de pago: SEMANAL, QUINCENAL, MENSAUAL o DIARIO (período de prueba)",
        "dia_canon": "Día de la semana en que vence la cuota (QUI = quincenal)",
        "cuota": "Valor de la cuota por ciclo",
        "no_canon": "Número de cuotas pactadas",
        "diassinpagar": "Días desde el último pago según el sistema (999 = sin pagos aún)",
        "justificacion": "Justificación o marca de gestión vigente ese día",
        "empresa": "Empresa del grupo",
    },
    "rentabilidad": {
        "id_contrato": "Código anónimo del contrato",
        "concepto": "Desenlace del episodio cerrado (NORMAL, PRONTO PAGO o tipo de carta)",
        "fecha_inicio": "Inicio del episodio",
        "fecha_final": "Fin del episodio",
        "fecha_registro": "Fecha en que se registró el cierre",
        "tipopago": "Frecuencia de pago",
        "dia_canon": "Día de vencimiento de la cuota",
        "cuota": "Valor de la cuota",
        "no_canon": "Número de cuotas pactadas",
        "fuente": "Tabla de origen (rentabilidad_activos o rentabilidad_activos_cartas)",
    },
}


def diccionario(nombre, df):
    filas = []
    for col in df.columns:
        s = df[col]
        vacio = s.isna() | s.astype(str).str.strip().isin(["", "nan"])
        if pd.api.types.is_numeric_dtype(s):
            rango = f"{s.min():,.2f} a {s.max():,.2f}"
        elif pd.api.types.is_datetime64_any_dtype(s):
            rango = f"{s.min().date()} a {s.max().date()}"
        elif col == "id_contrato":
            rango = f"{s.nunique():,d} códigos"
        else:
            vc = s[~vacio].value_counts()
            rango = ", ".join(map(str, vc.index[:6])) + (f" (+{len(vc) - 6})" if len(vc) > 6 else "")
        filas.append({"tabla": nombre, "columna": col, "tipo": str(s.dtype), "significado": SIGNIFICADO[nombre][col],
                      "% faltantes": round(vacio.mean() * 100, 1), "rango o categorías": rango})
    return pd.DataFrame(filas)


dicc = pd.concat([diccionario(n, d) for n, d in (("abonos", abonos), ("motos_diario", diario), ("rentabilidad", rentab))])
dicc.to_csv(DIR_SALIDA / "reports" / "diccionario_datos.csv", index=False, encoding="utf-8-sig")
dicc

**Lectura.** Los faltantes altos de `cartamotivo` y `justificacion` son esperables:
solo se llenan cuando hay carta o justificación. Las columnas que importan para el
modelo (fechas, montos, frecuencia, cuota) están completas; el diagnóstico de
faltantes reales se hace en la sección 9, sobre las variables ya construidas.

## 4. Cifras base

In [ ]:
motivo = abonos.cartamotivo.fillna("").str.strip().str.upper()
es_carta = abonos.carta.eq("SI")
pagos = pagos_por_dia(abonos)
cartas = (abonos[es_carta].assign(motivo=motivo[es_carta].replace("", "(sin motivo)"))
          .groupby("motivo").agg(filas=("id_contrato", "size"), contratos=("id_contrato", "nunique"))
          .sort_values("filas", ascending=False))
cartas["¿pago real?"] = np.where(cartas.index == "PRONTO PAGO", "sí", "no")
cartas["¿retiro?"] = np.where(cartas.index.isin(CARTAS_RETIRO), "sí", "no")

base = {
    "registros de pago (filas de abonos)": len(abonos),
    "días con pago real (contrato-día)": len(pagos),
    "contratos (motos) distintos": abonos.id_contrato.nunique(),
    "rango de fechas de pago": f"{abonos.fecha.min().date()} a {CORTE.date()}",
    "filas con carta": int(es_carta.sum()),
    "contratos con carta de retiro": int(abonos[es_carta & motivo.isin(CARTAS_RETIRO)].id_contrato.nunique()),
}
CIFRAS["base"] = base
display(pd.Series(base, name="valor").to_frame())
display(cartas)

**El número de contrato identifica la moto, no al cliente.** Cuando a un cliente se le
retira la moto, esta se asigna a otro con el mismo número. Por eso la unidad de análisis
es el **episodio**: un cliente con una moto, identificado por (`id_contrato`,
`fecha_inicio`) en la foto diaria.

In [ ]:
tp_diario = (diario.assign(fi_valida=diario.fecha_inicio.ne("0000-00-00"))
             .groupby(["id_contrato", "fecha_inicio"]).agg(tipopago=("tipopago", "first"), valida=("fi_valida", "first")))
episodios_por_contrato = diario.loc[diario.fecha_inicio.ne("0000-00-00")].groupby("id_contrato").fecha_inicio.nunique()
print("Tramos (contrato, fecha_inicio) en la foto diaria:", len(tp_diario))
print("  por tipo de pago:", tp_diario[tp_diario.valida].tipopago.value_counts().to_dict())
print("  con fecha_inicio 0000-00-00 (trámites pendientes, se descartan):", int((~tp_diario.valida).sum()))
print("Contratos con 2 o más fechas de inicio distintas:", int((episodios_por_contrato >= 2).sum()), "de", len(episodios_por_contrato))

Los tramos `DIARIO` son un período de prueba de alrededor de un mes sin contrato de
propiedad: si el cliente responde, pasa a pago semanal o quincenal con una
`fecha_inicio` nueva. Se descartan, al igual que los tramos con `fecha_inicio`
0000-00-00 (cliente asignado con trámites pendientes, sin la moto todavía).

In [ ]:
ep = construir_episodios(diario)
ciclos = construir_ciclos(ep, pagos, CORTE)
ep = asignar_retiro(ep, retiros(abonos))
ep = marcar_poblacion(ep, ciclos, K, H, X, CORTE, INICIO_COHORTE)
desde_cohorte = ep[ep.fecha_inicio >= INICIO_COHORTE]

frec = desde_cohorte.tipopago.value_counts()
emp = desde_cohorte.empresa.value_counts()
CIFRAS["episodios"] = {
    "episodios de leasing (todos)": len(ep),
    "episodios desde la cohorte": len(desde_cohorte),
    "semanal": int(frec.get("SEMANAL", 0)), "quincenal": int(frec.get("QUINCENAL", 0)),
    "motohoy": int(emp.get("MOTOHOY", 0)), "rentaya": int(emp.get("RENTAYA", 0)),
    "con retiro (desde la cohorte)": int(desde_cohorte.fecha_retiro.notna().sum()),
    "ciclos de pago esperados (todos)": len(ciclos),
}
pd.Series(CIFRAS["episodios"], name="valor").to_frame()

In [ ]:
just = diario.justificacion.fillna("").str.strip().str.upper()
grupo = pd.Series("no es justificación", index=just.index)
from variables import ENFERMEDAD, INMOVILIZACION, RETENCION, PROBLEMA
for nombre, valores in (("enfermedad o calamidad", ENFERMEDAD), ("inmovilización", INMOVILIZACION),
                        ("retención temporal (MOTO INGRESADA)", RETENCION), ("cliente problema", PROBLEMA)):
    grupo[just.isin(valores)] = nombre
d_j = diario[just.ne("")].assign(grupo=grupo[just.ne("")], just=just[just.ne("")]).sort_values(["id_contrato", "fecha_registro"])
# Un evento es un tramo de dias consecutivos con la misma justificacion
d_j["nuevo"] = (d_j.groupby(["id_contrato", "just"]).fecha_registro.diff().dt.days.ne(1))
tabla_j = d_j.groupby("grupo").agg(dias_registrados=("just", "size"), eventos=("nuevo", "sum"), contratos=("id_contrato", "nunique"))
CIFRAS["justificaciones"] = tabla_j.eventos.astype(int).to_dict()
tabla_j.sort_values("eventos", ascending=False)

## 5. Fecha esperada de pago

`np_his_abonos` no tiene una columna de fecha esperada. Se reconstruye con el
calendario de cada episodio: para pago semanal, cada `dia_canon` después de la fecha
de inicio; para quincenal, cada 14 días. La figura muestra en qué día caen los pagos
reales respecto al `dia_canon` del episodio vigente: la mitad cae exactamente ese día,
lo que confirma que es la fecha esperada.

In [ ]:
sem = ep[ep.tipopago.eq("SEMANAL") & ep.dia_canon_num.notna()]
m = pd.merge_asof(pagos.sort_values("fecha"), sem[["id_contrato", "fecha_inicio", "fin", "dia_canon_num"]].sort_values("fecha_inicio"),
                  left_on="fecha", right_on="fecha_inicio", by="id_contrato", direction="backward")
m = m[m.dia_canon_num.notna() & (m.fecha <= m.fin) & (m.fecha >= "2024-01-01")]
rel = ((m.fecha.dt.weekday - m.dia_canon_num) % 7).astype(int)
dist = rel.value_counts(normalize=True).sort_index() * 100
etiquetas = ["día del\ncanon", "+1", "+2", "+3", "+4", "+5", "+6"]
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(etiquetas, dist.values, color=AZUL, width=0.6)
for i, v in enumerate(dist.values):
    ax.text(i, v + 1, f"{v:.0f} %", ha="center", color=TINTA, fontsize=9)
ax.set_ylabel("% de los pagos")
ax.set_title("Día de pago respecto al día de canon (pago semanal)")
ax.set_ylim(0, dist.max() + 8)
fig.tight_layout(); fig.savefig(DIR_FIG / "dia_pago_vs_canon.png"); plt.show()

## 6. Consistencia entre multa y días de atraso

Cada día con pago (cualquier dinero entregado: abono, transferencia, multa o repuesto)
cubre un ciclo del calendario, o varios si el pago al canon alcanza varias cuotas. El
atraso de un ciclo es la diferencia entre la fecha de ese pago y la fecha esperada.

In [ ]:
c = ciclos[ciclos.pagado & (ciclos.fecha_pago >= "2024-01-01")].copy()
c["tramo"] = pd.cut(c.dias_atraso, [-1, 0, 2, 6, 13, 10**5], labels=["0", "1-2", "3-6", "7-13", "14+"])
consist = c.groupby("tramo", observed=True).agg(
    ciclos=("multa", "size"),
    sin_multa=("multa", lambda s: (s == 0).mean() * 100),
    multa_15=("multa", lambda s: (s == 15).mean() * 100),
    multa_25=("multa", lambda s: (s == 25).mean() * 100),
    otra=("multa", lambda s: (~s.isin([0, 15, 25])).mean() * 100),
).round(1)
rho = c[["multa", "dias_atraso"]].corr("spearman").iloc[0, 1]
CIFRAS["multa"] = {"spearman_multa_atraso": round(rho, 3), **{f"sin_multa_{t}": consist.loc[t, "sin_multa"] for t in consist.index}}
print(f"Correlación de Spearman multa ~ días de atraso: {rho:.3f}")
consist.rename(columns={"sin_multa": "% sin multa", "multa_15": "% multa 15", "multa_25": "% multa 25", "otra": "% otro valor"})

**Lectura.** La multa sigue escalones según el atraso dentro de la semana: sin atraso
casi nunca hay multa, con 1 a 2 días es de 15 mil y desde 3 días es de 25 mil. Desde
7 días la relación se debilita: un cliente que va una semana atrasado pero paga el
día del canon muchas veces no recibe multa. La multa mide el atraso **dentro de la
semana**, no el acumulado; por eso no es del todo redundante con el atraso, aunque sí
lo es con la proporción de ciclos atrasados (sección 8).

## 7. Distribución del atraso y elección de *k*, *H* y *X*

In [ ]:
pct = ciclos.dias_atraso.quantile([.5, .75, .9, .95, .99])
CIFRAS["atraso_ciclo_percentiles"] = pct.round(1).to_dict()
print("Percentiles del atraso por ciclo (días):", pct.round(1).to_dict())
fig, ax = plt.subplots(figsize=(6, 3))
v = ciclos.dias_atraso.clip(upper=60)
ax.hist(v, bins=np.arange(0, 62, 1), color=AZUL, rwidth=0.85)
ax.set_yscale("log")
ax.set_xlabel("Días de atraso del ciclo (60 = 60 o más)")
ax.set_ylabel("Ciclos (escala log)")
ax.set_title("Atraso por ciclo de pago")
fig.tight_layout(); fig.savefig(DIR_FIG / "atraso_por_ciclo.png"); plt.show()

**Elección de X.** Para cada episodio se toma el máximo atraso de un ciclo en la
ventana de predicción y se compara con el porcentaje que termina con la moto retirada.
El retiro es discrecional y no es la etiqueta, pero sirve para validar que el umbral
separa comportamientos de riesgo real.

In [ ]:
def max_atraso_ventana(pob, k, h):
    cc = ciclos[ciclos.id_episodio.isin(pob.id_episodio)].merge(pob[["id_episodio", "fecha_inicio"]])
    semana = (cc.fecha_esperada - cc.fecha_inicio).dt.days / 7
    cc = cc[(semana > k) & (semana <= k + h)]
    return cc.groupby("id_episodio").dias_atraso.max().reindex(pob.id_episodio).fillna(0)

pob = ep[ep.motivo_exclusion.isna()]
mx = max_atraso_ventana(pob, K, H).values
retiro_post = (pob.fecha_retiro > pob.fecha_inicio + pd.Timedelta(weeks=K)).values
tramos = pd.cut(mx, [-1, 0, 2, 6, 13, 20, 29, 59, 10**5], labels=["0", "1-2", "3-6", "7-13", "14-20", "21-29", "30-59", "60+"])
t_x = pd.DataFrame({"tramo": tramos, "retiro": retiro_post}).groupby("tramo", observed=True).agg(
    episodios=("retiro", "size"), pct_retiro=("retiro", lambda s: s.mean() * 100)).round(1)
CIFRAS["retiro_por_atraso_max"] = t_x.pct_retiro.to_dict()

fig, ax = plt.subplots(figsize=(6.5, 3.2))
colores = [NARANJA if t in ("14-20", "21-29", "30-59", "60+") else AZUL for t in t_x.index]
ax.bar(t_x.index.astype(str), t_x.pct_retiro, color=colores, width=0.6)
for i, (p, n) in enumerate(zip(t_x.pct_retiro, t_x.episodios)):
    ax.text(i, p + 1.5, f"{p:.0f} %", ha="center", color=TINTA, fontsize=9)
ax.set_xlabel(f"Máximo atraso de un ciclo en las semanas {K+1}-{K+H} (días)")
ax.set_ylabel("% retirados después de la semana k")
ax.set_title(f"El riesgo de retiro se estabiliza desde {X} días de atraso")
ax.set_ylim(0, t_x.pct_retiro.max() + 12)
fig.tight_layout(); fig.savefig(DIR_FIG / "retiro_vs_atraso_maximo.png"); plt.show()
t_x

**Elección de k y H.** La tabla compara combinaciones: tamaño de la población,
episodios retirados antes de poder predecir y prevalencia de la etiqueta. Un *k*
pequeño permite actuar antes y conserva más episodios, pero con pocos ciclos las
variables de variabilidad se estiman mal; un *k* grande pierde los retiros tempranos.

In [ ]:
filas = []
for k in (4, 6, 8, 12):
    for h in (8, 12, 26):
        e = marcar_poblacion(ep.drop(columns=["motivo_exclusion", "semanas", "cerrado"]), ciclos, k, h, X, CORTE, INICIO_COHORTE)
        p = e[e.motivo_exclusion.isna()]
        y_kh = etiqueta(p, ciclos, k, h, X)
        filas.append({"k": k, "H": h, "población": len(p),
                      "cerrados antes de k": int((e.motivo_exclusion == "cerrado antes de la semana k").sum()),
                      "sin desenlace aún": int((e.motivo_exclusion == "sin desenlace observable aun").sum()),
                      "% positivos": round(y_kh.mean() * 100, 1)})
rejilla = pd.DataFrame(filas)
rejilla.style.apply(lambda r: ["font-weight: bold" if (r.k == K and r.H == H) else "" for _ in r], axis=1)

**Decisión.** `K` = 8 semanas (8 ciclos semanales o 4 quincenales, suficientes para
medir regularidad y variabilidad), `H` = 12 semanas (el horizonte de 3 meses es el de
mejor desempeño en [3] y conserva episodios de 2026) y `X` = 14 días (dos cuotas
semanales sin pagar, donde el riesgo de retiro deja de crecer).

## 8. Población, etiqueta y corte temporal

```
fecha_inicio            t_k = inicio + K semanas              t_k + H semanas
     |------- ventana de observación -------|------- ventana de predicción -------|
       variables de entrada (solo fechas <= t_k)     etiqueta (ciclos que vencen aquí)
```

- **Variables de entrada:** solo información con fecha menor o igual a t_k. Un ciclo
  vencido antes de t_k y pagado después se ve como "sin pagar" con el atraso
  acumulado hasta t_k, que es lo que se sabría ese día (prueba en
  `tests/test_ciclos.py::test_variables_de_entrada_no_ven_informacion_posterior_a_k`).
- **Etiqueta (y = 1):** algún ciclo que vence en (t_k, t_k + H] alcanza X o más días de
  atraso, o nunca se pagó porque el episodio se cerró (la moto se retiró antes de
  llegar a X días). En otro caso y = 0.
- **Población:** episodios que siguen con la moto en t_k y cuyo desenlace ya es
  observable. El retiro y las marcas de gestión nunca forman parte de la etiqueta.

In [ ]:
embudo = ep.motivo_exclusion.fillna("EN LA POBLACIÓN").value_counts()
CIFRAS["embudo"] = embudo.to_dict()
display(embudo.to_frame("episodios"))

y = etiqueta(pob, ciclos, K, H, X)
cc = ciclos[ciclos.id_episodio.isin(pob.id_episodio)].merge(pob[["id_episodio", "fecha_inicio", "cerrado"]])
semana = (cc.fecha_esperada - cc.fecha_inicio).dt.days / 7
cc = cc[(semana > K) & (semana <= K + H)]
por_atraso = cc[cc.dias_atraso >= X].id_episodio.unique()
solo_nunca = set(y[y == 1].index) - set(por_atraso)
con_carta = pob.set_index("id_episodio").loc[list(solo_nunca)].fecha_retiro.notna().sum()
CIFRAS["etiqueta"] = {"poblacion": len(pob), "positivos": int(y.sum()), "pct_positivos": round(y.mean() * 100, 1),
                      "positivos_por_atraso": len(por_atraso), "positivos_solo_nunca_pagado": len(solo_nunca),
                      "de_esos_con_carta": int(con_carta), "razon_negativos_positivos": round((1 - y.mean()) / y.mean(), 2)}
pd.Series(CIFRAS["etiqueta"], name="valor").to_frame()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 2.4))
conteo = y.value_counts().sort_index()
ax.barh(["y = 0 (sin riesgo grave)", "y = 1 (riesgo grave)"], conteo.values, color=[AZUL, NARANJA], height=0.5)
for i, v in enumerate(conteo.values):
    ax.text(v + 20, i, f"{v:,d} ({v / conteo.sum() * 100:.1f} %)".replace(",", "."), va="center", color=TINTA, fontsize=9)
ax.set_xlim(0, conteo.max() * 1.35)
ax.set_xlabel("Episodios")
ax.set_title("Balance de clases")
ax.grid(axis="y", visible=False)
fig.tight_layout(); fig.savefig(DIR_FIG / "balance_clases.png"); plt.show()

anio = pob.fecha_inicio.dt.year
print("Prevalencia por año de inicio:", (y.groupby(anio.values).mean() * 100).round(1).to_dict())
print("Prevalencia por empresa:", (y.groupby(pob.empresa.values).mean() * 100).round(1).to_dict())

## 9. Variables candidatas de las primeras *k* semanas

Las variables se expresan como proporciones o promedios por ciclo, no como conteos
absolutos, para que sean comparables entre frecuencias. "Semanas observadas" es igual
a *k* para todos los episodios de la población, así que no se incluye como variable;
su equivalente por frecuencia es `ciclos_esperados`.

In [ ]:
Xv = variables_entrada(pob, ciclos, pagos, diario, K)
DESCRIPCION = {
    "ciclos_esperados": "Cuotas que vencieron en la ventana",
    "prop_ciclos_atrasados": "Proporción de ciclos con atraso >= 1 día",
    "prop_ciclos_atraso_3mas": "Proporción de ciclos con atraso >= 3 días",
    "atraso_promedio": "Días de atraso promedio por ciclo",
    "atraso_maximo": "Máximo atraso de un ciclo",
    "ciclos_sin_pagar_k": "Ciclos vencidos sin pagar en t_k",
    "dias_con_pago": "Días con algún pago",
    "dias_hasta_primer_pago": "Días entre el inicio y el primer pago",
    "pago_canon_total": "Abono + transferencia acumulados",
    "cv_pago_canon": "Coeficiente de variación del pago al canon",
    "multa_acumulada": "Multa acumulada",
    "prop_pagos_con_multa": "Proporción de pagos con multa",
    "repuesto_acumulado": "Dinero devuelto por préstamos",
    "pagos_por_ciclo": "Días con pago por ciclo esperado",
    "deficit_relativo": "1 - pago al canon / (cuota x ciclos)",
    "justif_enfermedad": "Eventos de justificación por enfermedad o calamidad",
    "justif_inmovilizacion": "Eventos de justificación por inmovilización",
    "retencion_temporal": "Eventos de moto ingresada (marca de gestión)",
    "cliente_problema": "Eventos de cliente problema (marca de gestión)",
    "tipopago": "Frecuencia de pago", "empresa": "Empresa", "cuota": "Valor de la cuota",
    "no_canon": "Cuotas pactadas", "dia_canon": "Día de vencimiento",
}
num = Xv.select_dtypes("number")
resumen = pd.DataFrame({
    "descripción": pd.Series(DESCRIPCION),
    "% faltantes": (Xv.isna().mean() * 100).round(1),
    "mediana": num.median().round(2),
    "media y=0": num[y.values == 0].mean().round(2),
    "media y=1": num[y.values == 1].mean().round(2),
    "Spearman con y": num.corrwith(y.astype(float), method="spearman").round(3),
}).loc[list(DESCRIPCION)]
resumen.to_csv(DIR_SALIDA / "reports" / "variables_candidatas.csv", encoding="utf-8-sig")
resumen

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
r = resumen["Spearman con y"].dropna().sort_values()
ax.barh(r.index, r.values, color=[NARANJA if v > 0 else AZUL for v in r.values], height=0.6)
ax.axvline(0, color=TINTA_2, lw=1)
ax.set_xlabel("Correlación de Spearman con y")
ax.set_title("Asociación de cada variable con la etiqueta")
ax.grid(axis="y", visible=False)
fig.tight_layout(); fig.savefig(DIR_FIG / "spearman_variables.png"); plt.show()

**Redundancia entre multa y atraso.**

In [ ]:
pares = [("prop_pagos_con_multa", "prop_ciclos_atrasados"), ("multa_acumulada", "atraso_promedio"),
         ("pagos_por_ciclo", "dias_con_pago"), ("atraso_maximo", "atraso_promedio"), ("deficit_relativo", "ciclos_sin_pagar_k")]
red = pd.DataFrame([{"variable A": a, "variable B": b, "Spearman": round(num[[a, b]].corr("spearman").iloc[0, 1], 3)} for a, b in pares])
CIFRAS["redundancia"] = {f"{a}~{b}": s for a, b, s in red.itertuples(index=False)}
red

## 10. Datos faltantes y estrategia de imputación

Faltantes reales observados en las fuentes y en las variables construidas:

In [ ]:
falt = Xv.isna().sum()
falt = falt[falt > 0]
sin_pago = Xv.dias_con_pago.eq(0)
diag = pd.DataFrame({
    "faltantes": falt,
    "% de la población": (falt / len(Xv) * 100).round(1),
    "coinciden con 'sin pagos en la ventana'": [int((Xv[c].isna() & sin_pago).sum()) for c in falt.index],
    "coinciden con 'un solo pago'": [int((Xv[c].isna() & Xv.dias_con_pago.eq(1)).sum()) for c in falt.index],
})
print("Episodios sin ningún pago en la ventana:", int(sin_pago.sum()), "| tasa de y=1 en ellos:", round(y[sin_pago.values].mean() * 100, 1), "%")
print("Episodios semanales sin dia_canon (excluidos):", CIFRAS["embudo"].get("sin dia_canon", 0))
print("Pagos con diassinpagar = 999 en la foto diaria:", int((diario.diassinpagar == 999).sum()))
CIFRAS["faltantes"] = falt.to_dict()
diag

**Estrategia.** Ningún faltante se rellena con un valor inventado sin dejar rastro:

| Caso | Causa | Tratamiento |
|---|---|---|
| `dias_hasta_primer_pago`, `prop_pagos_con_multa` | episodio sin ningún pago en la ventana | no son faltantes al azar: se imputa el valor que corresponde a "no pagó" (largo de la ventana; 0) **y** se agrega un indicador `sin_pagos_k` |
| `cv_pago_canon` | 0 o 1 pago: la variación no está definida | se imputa 0 con el indicador `sin_pagos_k` o `un_solo_pago`; el valor se fija con datos de entrenamiento |
| `dia_canon` ausente (18 episodios) | sin calendario posible | se excluyen (0,4 %); imputarlo sería inventar la fecha esperada |
| `diassinpagar` = 999 | código del sistema para "sin pagos aún" | se trata como faltante, no como 999 días |
| multa = 0 entre 2020 y 2022 | la multa no se registraba | fuera de la cohorte (inicia en 2024-03-30) |
| atraso de ciclos nunca pagados | episodio cerrado o activo al corte | no se imputa: es un atraso censurado (al menos esos días) |

Cualquier imputación que dependa de estadísticos (mediana, por ejemplo) se ajusta
solo con el conjunto de entrenamiento, dentro del pipeline de validación cruzada.

## 11. Codificación y balance de clases

- **Codificación:** `tipopago` (2 niveles) y `empresa` (2 niveles) se codifican como
  binarias; `dia_canon` (MAR, MIE, JUE, QUI) con one-hot. Las demás son numéricas y se
  escalan (estandarización) dentro del pipeline.
- **Balance:** la razón entre clases se muestra abajo. Es un desbalance moderado; se
  compararán ponderación de clases (`class_weight`), submuestreo y SMOTE **solo
  dentro de los pliegues de entrenamiento**, y se evaluará con métricas sensibles al
  desbalance (AUC, G-mean, exactitud balanceada, KS, Brier).

In [ ]:
print(f"y=1: {y.sum():,d} | y=0: {(y == 0).sum():,d} | razón 1:{CIFRAS['etiqueta']['razon_negativos_positivos']}")

## 12. Exportar cifras para el documento

In [ ]:
def a_json(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    return str(o)

CIFRAS["parametros"] = {"K": K, "H": H, "X": X, "inicio_cohorte": str(INICIO_COHORTE.date()), "corte": str(CORTE.date())}
(DIR_SALIDA / "reports" / "cifras_eda.json").write_text(json.dumps(CIFRAS, indent=2, ensure_ascii=False, default=a_json), encoding="utf-8")
Xv.join(y).join(pob.set_index("id_episodio").fecha_inicio).to_csv((DIR_SALIDA if EN_COLAB else DIR_DATOS) / "dataset_modelo.csv.gz")
print(f"Resultados guardados en {DIR_SALIDA / 'reports'}")

# PARTE 2 · Etapa 1: perfiles de pago (clustering)

<sub>Contenido del cuaderno `02_clustering.ipynb`.</sub>

# Entregable II — Etapa 1: perfiles de pago (clustering)

Modelos y Simulación de Sistemas II — Universidad de Antioquia — Grupo 14

Este notebook construye la **etapa no supervisada** del sistema: agrupar los episodios
según su comportamiento de pago en las primeras *K* = 8 semanas. El perfil resultante
alimenta la etapa supervisada (`03_clasificacion`).

**Dos cuidados metodológicos** (retroalimentación del Entregable I):

1. **Sin fuga entre etapas.** El escalado, la imputación y el clustering se ajustan solo
   con los episodios de entrenamiento. A los de prueba únicamente se les *asigna* un
   perfil con el modelo ya entrenado (`predict`/`transform`, nunca `fit`).
2. **Validación temporal.** Los episodios se ordenan por `fecha_inicio`: los más antiguos
   entrenan y el 20 % más reciente prueba. Se purgan del entrenamiento los episodios cuya
   etiqueta se resuelve después de que empieza la prueba (`src/particion.py`).

La etiqueta `y` **no se usa para formar los perfiles ni para elegir *K***; solo se usa
después, para describir el riesgo de cada perfil. La explicación de cada variable y su
tratamiento están en `01_eda.ipynb` y en `reports/diccionario_datos.csv`.

In [ ]:
#@title ⚙️ Preparación de esta parte (no hace falta tocar)
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

EN_COLAB = "google.colab" in sys.modules
NOMBRE_CARPETA = "ProyectoModelos2_Colab"


def buscar_carpeta_drive():
    raices = [Path("/content/drive/MyDrive"), *Path("/content/drive").glob(".shortcut-targets-by-id/*")]
    for patron in (NOMBRE_CARPETA, f"*/{NOMBRE_CARPETA}", f"*/*/{NOMBRE_CARPETA}"):
        for raiz in raices:
            for carpeta in raiz.glob(patron):
                if (carpeta / "modelado.py").exists():
                    return carpeta
    return None


if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    carpeta = buscar_carpeta_drive()
    if carpeta is None:
        raise FileNotFoundError(
            f"No se encontro la carpeta {NOMBRE_CARPETA} en su Drive. Agreguele un acceso directo "
            "en Mi unidad (Compartidos conmigo > clic derecho > Organizar > Agregar acceso directo) "
            "y vuelva a ejecutar todo.")
    os.chdir(carpeta)
RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DIR_SALIDA = Path("/content/salidas") if EN_COLAB else RAIZ


def ubicar(archivos, carpetas):
    for c in carpetas:
        if all((c / a).exists() for a in archivos):
            return c
    raise FileNotFoundError(f"No se encontraron {archivos} en {[str(c) for c in carpetas]}.")


sys.path.insert(0, str(ubicar(["modelado.py", "particion.py"], [RAIZ / "src", Path.cwd()])))
DIR_DATOS = ubicar(["dataset_modelo.csv.gz"], [RAIZ / "data" / "anon", Path.cwd()])
DIR_FIG = DIR_SALIDA / "reports" / "figuras"
DIR_FIG.mkdir(parents=True, exist_ok=True)

from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.mixture import GaussianMixture

from modelado import COLS_PERFIL, Perfil
from particion import corte_por_fraccion, dividir_temporal, pliegues_expansivos

TINTA, TINTA_2, REJILLA = "#0b0b0b", "#52514e", "#e6e5e0"
AZUL, NARANJA = "#2a78d6", "#eb6834"
# Perfiles ordenados de menor a mayor riesgo: rampa de un solo matiz
COLORES = ["#f2c14e", "#f78154", "#d1495b", "#6b2737", "#3d1a24", "#1c0d12"]
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 10,
    "axes.edgecolor": TINTA_2, "axes.labelcolor": TINTA, "xtick.color": TINTA_2, "ytick.color": TINTA_2,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": REJILLA,
    "axes.axisbelow": True, "axes.titlesize": 11, "axes.titleweight": "bold",
})
CIFRAS = {}
SEMILLA = 0

## 1. Datos y partición temporal

Parámetros del estudio (los mismos del EDA): ventana de observación `K` = 8 semanas,
horizonte `H` = 12 semanas, atraso grave `X` = 14 días. La purga de la partición depende
de estos tres valores.

In [ ]:
K, H, X = 8, 12, 14
d = pd.read_csv(DIR_DATOS / "dataset_modelo.csv.gz", parse_dates=["fecha_inicio"])
corte = corte_por_fraccion(d.fecha_inicio, 0.2)
i_tr, i_te = dividir_temporal(d.fecha_inicio, K, H, X, corte)
A, B = d.iloc[i_tr].reset_index(drop=True), d.iloc[i_te].reset_index(drop=True)
purgados = int((d.fecha_inicio < corte).sum() - len(A))

resumen = pd.DataFrame({
    "episodios": [len(A), len(B)],
    "inicio desde": [A.fecha_inicio.min().date(), B.fecha_inicio.min().date()],
    "inicio hasta": [A.fecha_inicio.max().date(), B.fecha_inicio.max().date()],
    "% riesgo grave": [round(100 * A.y.mean(), 1), round(100 * B.y.mean(), 1)],
}, index=["entrenamiento", "prueba"])
display(resumen)
print(f"Fecha de corte: {corte.date()}. Purgados del entrenamiento: {purgados} episodios "
      "(su etiqueta se resuelve despues del corte).")
CIFRAS["particion"] = {"corte": str(corte.date()), "train": len(A), "test": len(B), "purgados": purgados,
                       "riesgo_train": round(float(A.y.mean()), 4), "riesgo_test": round(float(B.y.mean()), 4)}

El riesgo grave es mayor en los episodios recientes que en los antiguos: la cartera se
deterioró con el tiempo. Una partición aleatoria habría ocultado ese cambio, y por eso
las métricas de este notebook y del siguiente serán más exigentes (y más honestas) que
las de una validación cruzada sin orden temporal.

## 2. Elección del número de perfiles *K*

Se comparan de 2 a 7 perfiles con tres criterios, **todos calculados sin la etiqueta**:

- **Codo** (inercia en entrenamiento).
- **Silueta**, medida en entrenamiento y en validación temporal (pliegues de ventana
  expansiva; en cada pliegue el modelo se ajusta solo con lo anterior y se evalúa en el
  tramo siguiente).
- **Estabilidad**: se ajusta el clustering con la primera y con la segunda mitad (en el
  tiempo) del entrenamiento y se mide el acuerdo entre ambas asignaciones sobre los mismos
  episodios con el índice de Rand ajustado (ARI). Un valor cercano a 1 indica que los
  perfiles no dependen del periodo usado.

In [ ]:
pliegues = pliegues_expansivos(d.fecha_inicio, K, H, X, n_splits=4, fecha_max=corte)
mitad = len(A) // 2
filas = []
for k in range(2, 8):
    p = Perfil(k, seed=SEMILLA).fit(A)
    Z_tr = p.prep_.transform(A[p.cols_])
    sil_tr = silhouette_score(Z_tr, p.predict(A))
    sil_val = []
    for tr, va in pliegues:
        if len(tr) < 50:
            continue
        pf = Perfil(k, seed=SEMILLA).fit(d.iloc[tr])
        sil_val.append(silhouette_score(pf.prep_.transform(d.iloc[va][pf.cols_]), pf.predict(d.iloc[va])))
    a1 = Perfil(k, seed=SEMILLA).fit(A.iloc[:mitad]).predict(B)
    a2 = Perfil(k, seed=SEMILLA).fit(A.iloc[mitad:]).predict(B)
    filas.append({"K": k, "inercia": p.km_.inertia_, "silueta train": sil_tr,
                  "silueta validación": np.mean(sil_val), "estabilidad (ARI)": adjusted_rand_score(a1, a2)})
sel = pd.DataFrame(filas).set_index("K")
display(sel.round(3))

fig, ax = plt.subplots(1, 3, figsize=(11, 3.2))
ax[0].plot(sel.index, sel.inercia, marker="o", color=AZUL); ax[0].set_title("Codo"); ax[0].set_ylabel("inercia")
ax[1].plot(sel.index, sel["silueta train"], marker="o", color=AZUL, label="entrenamiento")
ax[1].plot(sel.index, sel["silueta validación"], marker="o", color=NARANJA, label="validación temporal")
ax[1].set_title("Silueta"); ax[1].legend(frameon=False)
ax[2].plot(sel.index, sel["estabilidad (ARI)"], marker="o", color=AZUL); ax[2].set_title("Estabilidad (ARI)")
for a in ax: a.set_xlabel("número de perfiles K")
fig.tight_layout(); fig.savefig(DIR_FIG / "seleccion_k.png"); plt.show()
CIFRAS["seleccion_k"] = sel.round(4).to_dict(orient="index")

**Lectura.** La silueta y la estabilidad son máximas con 2 perfiles, que solo separan a
quienes pagan bien de quienes no. Con 3 y 4 perfiles la estabilidad sigue siendo buena
(ARI cercano o superior a 0,8 con 4), pero cae con 5 o más: son perfiles que cambian según
el periodo con que se ajusten. Se elige ***K* = 4** como el mayor número de perfiles que se
mantiene estable, porque permite diferenciar niveles de riesgo útiles para el cobro (ver
sección 4). Esta decisión se tomó con criterios no supervisados, antes de mirar el riesgo
por perfil.

In [ ]:
KP = 4
modelo = Perfil(KP, seed=SEMILLA).fit(A)
A["perfil"] = modelo.predict(A)
B["perfil"] = modelo.predict(B)          # solo predict: la prueba no participa en el ajuste
Z_tr = modelo.prep_.transform(A[modelo.cols_])
Z_te = modelo.prep_.transform(B[modelo.cols_])

## 3. Comparación de algoritmos

Se comparan tres algoritmos vistos en el curso con el mismo *K* = 4 y el mismo
preprocesamiento: **k-means**, **jerárquico aglomerativo (Ward)** y **mezcla de
gaussianas (GMM)**. El jerárquico no tiene un `predict`; para asignar perfiles a la
prueba se usa el centroide más cercano de los grupos formados en entrenamiento. En todos
los casos los perfiles se numeran de menor a mayor atraso promedio.

In [ ]:
def ordenar_por_atraso(etiquetas, Z, cols):
    j = cols.index("atraso_promedio")
    medias = {c: Z[etiquetas == c, j].mean() for c in np.unique(etiquetas)}
    mapa = {c: i for i, c in enumerate(sorted(medias, key=medias.get))}
    return np.array([mapa[c] for c in etiquetas]), mapa

cols = modelo.cols_
# k-means
et_km_tr, et_km_te = A.perfil.to_numpy(), B.perfil.to_numpy()
# jerarquico (Ward)
h = AgglomerativeClustering(n_clusters=KP, linkage="ward").fit(Z_tr)
et_h_tr, mapa_h = ordenar_por_atraso(h.labels_, Z_tr, cols)
cent_h = np.vstack([Z_tr[et_h_tr == c].mean(axis=0) for c in range(KP)])
et_h_te = ((Z_te[:, None, :] - cent_h[None]) ** 2).sum(axis=2).argmin(axis=1)
# GMM
g = GaussianMixture(KP, covariance_type="full", n_init=5, random_state=SEMILLA).fit(Z_tr)
et_g_tr, mapa_g = ordenar_por_atraso(g.predict(Z_tr), Z_tr, cols)
et_g_te = np.array([mapa_g[c] for c in g.predict(Z_te)])

algs = {"k-means": (et_km_tr, et_km_te), "jerárquico (Ward)": (et_h_tr, et_h_te), "GMM": (et_g_tr, et_g_te)}
filas = []
for nombre, (e_tr, e_te) in algs.items():
    r_tr = pd.Series(A.y.to_numpy()).groupby(e_tr).mean()
    r_te = pd.Series(B.y.to_numpy()).groupby(e_te).mean()
    filas.append({
        "algoritmo": nombre,
        "silueta train": silhouette_score(Z_tr, e_tr),
        "silueta prueba": silhouette_score(Z_te, e_te),
        "ARI vs k-means (train)": adjusted_rand_score(et_km_tr, e_tr),
        "riesgo por perfil, train (%)": " / ".join(f"{100*v:.0f}" for v in r_tr),
        "riesgo por perfil, prueba (%)": " / ".join(f"{100*r_te.get(i, np.nan):.0f}" for i in range(KP)),
        "tamaños, prueba": " / ".join(str(int((e_te == i).sum())) for i in range(KP)),
    })
comp = pd.DataFrame(filas).set_index("algoritmo")
display(comp.round(3))
CIFRAS["comparacion_algoritmos"] = comp.round(4).to_dict(orient="index")

Se usa **k-means** para la etapa supervisada: es el más simple, el único de los tres con
`predict` nativo y sus centroides son interpretables. Los tres ordenan el riesgo de menor a
mayor en entrenamiento; el jerárquico y el GMM coinciden solo en parte con k-means (ARI
entre 0,6 y 0,7) y tienen una silueta menor. Además, el GMM no asigna a ningún episodio de
prueba al perfil más grave, y en k-means ese perfil tiene apenas 15: su porcentaje de riesgo
en prueba es poco preciso.

## 4. Caracterización de los perfiles

In [ ]:
perfil_nombre = {0: "Puntual", 1: "Atrasos leves", 2: "Se va quedando", 3: "No paga"}
tabla = pd.DataFrame({
    "episodios (train)": A.perfil.value_counts().sort_index(),
    "episodios (prueba)": B.perfil.value_counts().sort_index(),
    "% riesgo grave (train)": (100 * A.groupby("perfil").y.mean()).round(1),
    "% riesgo grave (prueba)": (100 * B.groupby("perfil").y.mean()).round(1),
}).fillna(0)
tabla.index = [f"{i} · {perfil_nombre[i]}" for i in tabla.index]
display(tabla)

medias = A.groupby("perfil")[COLS_PERFIL].mean().round(2)
medias.index = [f"{i} · {perfil_nombre[i]}" for i in medias.index]
display(medias.T)
CIFRAS["perfiles"] = {"tabla": tabla.to_dict(orient="index"), "medias": medias.to_dict(orient="index")}

fig, ax = plt.subplots(figsize=(6.5, 3.4))
x = np.arange(KP); w = 0.38
ax.bar(x - w/2, tabla["% riesgo grave (train)"], w, color=AZUL, label="entrenamiento")
ax.bar(x + w/2, tabla["% riesgo grave (prueba)"], w, color=NARANJA, label="prueba (más reciente)")
ax.set_xticks(x); ax.set_xticklabels(tabla.index, rotation=12); ax.set_ylabel("% con riesgo grave")
ax.set_title("Riesgo grave por perfil"); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(DIR_FIG / "riesgo_por_perfil.png"); plt.show()

Los nombres de los perfiles son una descripción de las medias de la tabla, no una
variable del modelo. Al comparar entrenamiento y prueba hay que tener presente que los
episodios recientes son más riesgosos y que un perfil con pocos episodios en prueba da un
porcentaje poco preciso.

## 5. Visualización: PCA y UMAP

Ambas proyecciones se ajustan **solo con entrenamiento**; la prueba se proyecta con el
modelo ya ajustado.

In [ ]:
pca = PCA(n_components=2, random_state=SEMILLA).fit(Z_tr)
P_tr, P_te = pca.transform(Z_tr), pca.transform(Z_te)
try:
    import umap
    um = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.2, random_state=SEMILLA).fit(Z_tr)
    U_tr, U_te = um.embedding_, um.transform(Z_te)
    proy = {"PCA": (P_tr, P_te), "UMAP": (U_tr, U_te)}
except ImportError:
    print("umap-learn no esta instalado: solo se muestra PCA (pip install umap-learn).")
    proy = {"PCA": (P_tr, P_te)}

fig, ax = plt.subplots(1, len(proy), figsize=(5.6 * len(proy), 4.4), squeeze=False)
for a, (nombre, (p_tr, p_te)) in zip(ax[0], proy.items()):
    for c in range(KP):
        m = A.perfil.to_numpy() == c
        a.scatter(p_tr[m, 0], p_tr[m, 1], s=7, color=COLORES[c], alpha=0.6, label=f"{c} · {perfil_nombre[c]}")
    a.scatter(p_te[:, 0], p_te[:, 1], s=14, facecolors="none", edgecolors=TINTA_2, linewidths=0.5, label="prueba")
    a.set_title(nombre); a.set_xlabel("componente 1"); a.set_ylabel("componente 2")
ax[0][0].legend(frameon=False, fontsize=8, markerscale=2)
fig.tight_layout(); fig.savefig(DIR_FIG / "proyeccion_perfiles.png"); plt.show()
print("Varianza explicada por las 2 primeras componentes (PCA):", pca.explained_variance_ratio_.round(3))
CIFRAS["pca_var_explicada"] = pca.explained_variance_ratio_.round(4).tolist()

## 6. Resumen y salida para la etapa supervisada

- Se eligió *K* = 4 perfiles con k-means, por estabilidad y utilidad, sin usar la etiqueta.
- El modelo de perfiles (`Perfil`, en `src/modelado.py`) se ajustará **dentro de cada
  pliegue** del notebook 03, sin ver nunca los datos de validación ni de prueba.

In [ ]:
(DIR_SALIDA / "reports" / "cifras_clustering.json").write_text(
    json.dumps(CIFRAS, indent=2, ensure_ascii=False, default=lambda o: float(o) if hasattr(o, "item") else str(o)),
    encoding="utf-8")
print("Resultados guardados en", DIR_SALIDA / "reports")

# PARTE 3 · Etapa 2: clasificación del riesgo grave

<sub>Contenido del cuaderno `03_clasificacion.ipynb`.</sub>

# Entregable II — Etapa 2: clasificación del riesgo grave de mora

Modelos y Simulación de Sistemas II — Universidad de Antioquia — Grupo 14

Este notebook entrena y compara clasificadores que predicen si un episodio caerá en
**riesgo grave de mora** (algún ciclo del horizonte con 14 o más días de atraso, o
cierre con carta de retiro por mal comportamiento) a partir de su comportamiento en las
primeras *K* semanas y, opcionalmente, de su perfil de pago (etapa 1, `02_clustering`).

**Protocolo de evaluación** (respuesta a la retroalimentación del Entregable I):

- **Validación temporal.** Los episodios se ordenan por `fecha_inicio`. El 20 % más
  reciente es la **prueba** y se usa una sola vez al final; el resto es **desarrollo**,
  donde se valida con pliegues de ventana expansiva (se entrena con lo anterior y se valida
  en el tramo siguiente). Se purgan los episodios cuya etiqueta se resuelve después del
  inicio de la prueba o del pliegue.
- **Sin fuga entre etapas.** Imputación, escalado, codificación y el clustering de perfiles
  viven dentro del `Pipeline` y se reajustan en cada pliegue solo con sus datos de
  entrenamiento. Los modelos se **seleccionan con la validación**, nunca con la prueba.
- **Umbral de decisión.** Se elige el que maximiza la G-mean sobre las predicciones de
  validación y se aplica sin cambios a la prueba.
- **Métricas.** AUC-ROC (no depende del umbral), F1 por clase, exactitud balanceada
  (BACC), G-mean y MCC, que son las del curso para clases desbalanceadas.

La etiqueta, las variables y su tratamiento están descritos en `01_eda.ipynb`.

In [ ]:
#@title ⚙️ Preparación de esta parte (no hace falta tocar)
import json
import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
EN_COLAB = "google.colab" in sys.modules
NOMBRE_CARPETA = "ProyectoModelos2_Colab"


def buscar_carpeta_drive():
    raices = [Path("/content/drive/MyDrive"), *Path("/content/drive").glob(".shortcut-targets-by-id/*")]
    for patron in (NOMBRE_CARPETA, f"*/{NOMBRE_CARPETA}", f"*/*/{NOMBRE_CARPETA}"):
        for raiz in raices:
            for carpeta in raiz.glob(patron):
                if (carpeta / "clasificacion.py").exists():
                    return carpeta
    return None


if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    carpeta = buscar_carpeta_drive()
    if carpeta is None:
        raise FileNotFoundError(
            f"No se encontro la carpeta {NOMBRE_CARPETA} en su Drive. Agreguele un acceso directo "
            "en Mi unidad (Compartidos conmigo > clic derecho > Organizar > Agregar acceso directo) "
            "y vuelva a ejecutar todo.")
    os.chdir(carpeta)
    get_ipython().system("pip install -q imbalanced-learn")
RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DIR_SALIDA = Path("/content/salidas") if EN_COLAB else RAIZ


def ubicar(archivos, carpetas):
    for c in carpetas:
        if all((c / a).exists() for a in archivos):
            return c
    raise FileNotFoundError(f"No se encontraron {archivos} en {[str(c) for c in carpetas]}.")


sys.path.insert(0, str(ubicar(["clasificacion.py", "ciclos.py"], [RAIZ / "src", Path.cwd()])))
DIR_DATOS = ubicar(["abonos.csv.gz", "motos_diario.csv.gz"], [RAIZ / "data" / "anon", Path.cwd()])
DIR_FIG = DIR_SALIDA / "reports" / "figuras"
DIR_FIG.mkdir(parents=True, exist_ok=True)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import confusion_matrix, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from clasificacion import (CAT, MARCAS, NUM_BASE, Preprocesador, armar, construir_dataset,
                           cargar_base, evaluar, metricas, modelos)
from particion import corte_por_fraccion, dividir_temporal

TINTA, TINTA_2, REJILLA = "#0b0b0b", "#52514e", "#e6e5e0"
AZUL, NARANJA, VERDE = "#2a78d6", "#eb6834", "#2a9d6f"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 10,
    "axes.edgecolor": TINTA_2, "axes.labelcolor": TINTA, "xtick.color": TINTA_2, "ytick.color": TINTA_2,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": REJILLA,
    "axes.axisbelow": True, "axes.titlesize": 11, "axes.titleweight": "bold",
})
CIFRAS = {}
SEMILLA = 0
INICIO_COHORTE = pd.Timestamp("2024-03-30")

## 1. Datos

Se reconstruyen episodios, ciclos de pago y variables a partir de la copia anonimizada.
Parámetros base: `K` = 8 semanas de observación, `H` = 12 de predicción y `X` = 14 días de
atraso. En la sección 5 se varían los tres.

In [ ]:
ep, ciclos, pagos, diario, CORTE = cargar_base(DIR_DATOS)
K, H, X = 8, 12, 14
data = construir_dataset(ep, ciclos, pagos, diario, CORTE, INICIO_COHORTE, K, H, X)
COLS = NUM_BASE + MARCAS + CAT            # incluye las variables que usa el perfil
COLS_SIN_MARCAS = [c for c in COLS if c not in MARCAS]

corte = corte_por_fraccion(data.fecha_inicio, 0.2)
i_dev, i_te = dividir_temporal(data.fecha_inicio, K, H, X, corte)
print(f"Episodios: {len(data)} | positivos: {100 * data.y.mean():.1f} %")
print(f"Corte temporal: {corte.date()} | desarrollo: {len(i_dev)} ({100 * data.y.iloc[i_dev].mean():.1f} % positivos)"
      f" | prueba: {len(i_te)} ({100 * data.y.iloc[i_te].mean():.1f} % positivos)")
CIFRAS["datos"] = {"n": len(data), "pos": round(float(data.y.mean()), 4), "corte": str(corte.date()),
                   "n_dev": len(i_dev), "n_prueba": len(i_te)}

## 2. Comparación de modelos

Seis clasificadores del curso, todos con el perfil de pago y las marcas de gestión como
entradas y con ponderación de clases cuando el modelo la admite (regresión logística,
árbol, Random Forest y SVM). Los modelos se ordenan por su **AUC de validación**
(media de los pliegues temporales); la prueba se muestra para informar, no para elegir.

In [ ]:
resultados = {}
for nombre, clf in modelos(SEMILLA).items():
    resultados[nombre] = evaluar(armar(clf, con_perfil=True, con_marcas=True, seed=SEMILLA), data, COLS, K, H, X)

def tabla_resultados(res):
    filas = []
    for nombre, r in res.items():
        fila = {"modelo": nombre, "AUC validación": f"{r['auc_validacion']:.3f} ± {r['auc_validacion_sd']:.3f}"}
        fila.update({f"{k} (prueba)": round(v, 3) for k, v in r["prueba"].items() if k != "exactitud"})
        fila["umbral"] = round(r["umbral"], 2)
        filas.append(fila)
    t = pd.DataFrame(filas).set_index("modelo")
    orden = sorted(res, key=lambda n: -res[n]["auc_validacion"])
    return t.loc[orden]

comp = tabla_resultados(resultados)
display(comp)
MEJOR = comp.index[0]
print("Mejor modelo por AUC de validación:", MEJOR)
CIFRAS["modelos"] = comp.to_dict(orient="index")

**Cómo leer la tabla.** El AUC de validación es el promedio de cuatro pliegues temporales
(± su desviación); un valor alto y estable indica un modelo que generaliza hacia periodos
posteriores. Las columnas de prueba usan el umbral elegido en validación, así que miden
también qué tan bien se transfiere ese umbral a un periodo más reciente y más riesgoso.

**Lectura.** La regresión logística es la mejor en validación (AUC 0,805) y el Random Forest
la sigue de cerca (0,798) y es el mejor en prueba (0,784). Esas diferencias están dentro de
la desviación entre pliegues (≈ 0,02), así que no hay un ganador claro entre los dos; el
árbol de decisión es el único claramente peor (0,741 en validación). Todos los modelos
bajan en prueba frente a validación porque los episodios recientes son más riesgosos.

## 3. ¿Aporta el perfil de la etapa 1? ¿Y las marcas de gestión?

Se compara el mismo modelo con y sin el perfil de pago, y con y sin las marcas de gestión
de la empresa (`MOTO INGRESADA` y `cliente problema`), que son decisiones del cobrador y
podrían no estar disponibles en todos los casos de uso. Se repite para tres modelos.

In [ ]:
filas = []
for nombre in ["Regresión logística", "Random Forest", "Gradient Boosting"]:
    clf = modelos(SEMILLA)[nombre]
    for perfil in (False, True):
        for marcas in (True, False):
            cols = COLS if marcas else COLS_SIN_MARCAS
            r = evaluar(armar(clf, perfil, marcas, SEMILLA), data, cols, K, H, X)
            filas.append({"modelo": nombre, "perfil": "con" if perfil else "sin", "marcas": "con" if marcas else "sin",
                          "AUC validación": round(r["auc_validacion"], 3), "AUC prueba": round(r["prueba"]["AUC"], 3),
                          "BACC prueba": round(r["prueba"]["BACC"], 3), "MCC prueba": round(r["prueba"]["MCC"], 3)})
ablacion = pd.DataFrame(filas)
display(ablacion)
CIFRAS["ablacion"] = ablacion.to_dict(orient="records")

**Lectura.** Ni el perfil de la etapa 1 ni las marcas de gestión mejoran el desempeño: las
diferencias en AUC (≤ 0,004) son mucho menores que la variabilidad entre pliegues (≈ 0,02).
Para la regresión logística el AUC de validación es incluso un poco mayor sin marcas
(0,809 frente a 0,805). Dos consecuencias: (1) el modelo no depende de decisiones
discrecionales del cobrador, lo que lo hace aplicable en cualquier momento; (2) las
variables de comportamiento ya contienen la información que resume el perfil, que es
coherente con que el clustering se construye con esas mismas variables. El perfil sigue
siendo útil como herramienta de **descripción e interpretación** (notebook 02), aunque no
como entrada adicional del clasificador.

## 4. Estrategias de balance de clases

La clase positiva es el 36 %, un desbalance moderado. Se comparan cuatro tratamientos con
la regresión logística y el Random Forest: sin tratamiento, ponderación de clases,
submuestreo aleatorio y SMOTE. **El remuestreo se hace solo con los datos de entrenamiento
de cada pliegue** (el `Pipeline` de imbalanced-learn lo aplica solo en `fit`).

In [ ]:
def con_balance(clf, estrategia):
    pre = Preprocesador(con_perfil=True, con_marcas=True, seed=SEMILLA)
    if estrategia == "ninguno":
        return ImbPipeline([("pre", pre), ("clf", clf)])
    if estrategia == "ponderación":
        clf = clf.set_params(class_weight="balanced")
        return ImbPipeline([("pre", pre), ("clf", clf)])
    muestreador = RandomUnderSampler(random_state=SEMILLA) if estrategia == "submuestreo" else SMOTE(random_state=SEMILLA)
    return ImbPipeline([("pre", pre), ("mues", muestreador), ("clf", clf)])

base = {
    "Regresión logística": LogisticRegression(C=0.5, max_iter=2000),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=10, n_jobs=-1, random_state=SEMILLA),
}
filas = []
for nombre, clf in base.items():
    for est in ("ninguno", "ponderación", "submuestreo", "SMOTE"):
        r = evaluar(con_balance(clf, est), data, COLS, K, H, X)
        filas.append({"modelo": nombre, "balance": est, "AUC validación": round(r["auc_validacion"], 3),
                      "AUC prueba": round(r["prueba"]["AUC"], 3), "F1 clase 1 (prueba)": round(r["prueba"]["F1 (clase 1)"], 3),
                      "BACC prueba": round(r["prueba"]["BACC"], 3), "MCC prueba": round(r["prueba"]["MCC"], 3)})
balance = pd.DataFrame(filas)
display(balance)
CIFRAS["balance"] = balance.to_dict(orient="records")

**Lectura.** Con AUC no hay diferencias entre estrategias de balance, como era de esperar:
el AUC mide el orden de las probabilidades y el remuestreo cambia sobre todo dónde cae el
umbral. En el Random Forest el submuestreo mejora algo la BACC (0,717) y el MCC (0,449) en
prueba, pero su AUC de validación no mejora (0,797), así que no hay evidencia suficiente para
preferirlo. Con un desbalance moderado como el de este problema (35 % de positivos), la
ponderación de clases o no tratar el desbalance es una opción simple y suficiente.

## 5. Ventana de observación, horizonte y umbral de atraso

La utilidad operativa del modelo depende de cuánto historial se necesita (*K*), de qué tan
lejos se predice (*H*) y de qué se llama "atraso grave" (*X*). Se evalúan todas las
combinaciones de *K* ∈ {4, 6, 8, 12} semanas, *H* ∈ {8, 12, 16} semanas y *X* ∈ {7, 14, 21}
días con una regresión logística (con perfil y marcas) y la misma validación temporal.

**Precauciones.** Cada combinación define una población y una etiqueta distintas (un *K* o
*H* mayor exige episodios más largos y exige más tiempo para observar el desenlace), así
que se reporta el tamaño y la proporción de positivos junto al desempeño, y los AUC entre
filas con distinto *X* **no son estrictamente comparables**: miden tareas diferentes. El
*X* = 14 días queda como la definición base del estudio; los demás valores son un análisis
de sensibilidad y no sustituyen esa definición.

In [ ]:
archivo_rejilla = DIR_SALIDA / "reports" / "rejilla_khx.csv"
if archivo_rejilla.exists():
    rejilla = pd.read_csv(archivo_rejilla)
else:
    filas = []
    for k in (4, 6, 8, 12):
        for h in (8, 12, 16):
            for x in (7, 14, 21):
                dat = construir_dataset(ep, ciclos, pagos, diario, CORTE, INICIO_COHORTE, k, h, x)
                if dat.y.nunique() < 2 or len(dat) < 300:
                    continue
                r = evaluar(armar(LogisticRegression(C=0.5, max_iter=2000, class_weight="balanced"), True, True, SEMILLA),
                            dat, COLS, k, h, x)
                filas.append({"K": k, "H": h, "X": x, "episodios": len(dat), "% positivos": round(100 * dat.y.mean(), 1),
                              "n prueba": r["n_prueba"], "AUC validación": round(r["auc_validacion"], 3),
                              "AUC prueba": round(r["prueba"]["AUC"], 3), "BACC prueba": round(r["prueba"]["BACC"], 3),
                              "MCC prueba": round(r["prueba"]["MCC"], 3)})
    rejilla = pd.DataFrame(filas)
    rejilla.to_csv(archivo_rejilla, index=False)
display(rejilla)
CIFRAS["rejilla"] = rejilla.to_dict(orient="records")

fig, ax = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
for a, x in zip(ax, (7, 14, 21)):
    for h, col in zip((8, 12, 16), (AZUL, NARANJA, VERDE)):
        s = rejilla[(rejilla.X == x) & (rejilla.H == h)].sort_values("K")
        a.plot(s.K, s["AUC validación"], marker="o", color=col, label=f"H = {h}")
    a.set_title(f"X = {x} días"); a.set_xlabel("K (semanas de observación)"); a.set_xticks([4, 6, 8, 12])
ax[0].set_ylabel("AUC de validación temporal"); ax[0].legend(frameon=False)
fig.tight_layout(); fig.savefig(DIR_FIG / "rejilla_khx.png"); plt.show()

**Lectura.** Tres hallazgos, a leer con las precauciones indicadas arriba:

- **Se predice igual de bien con menos historial.** Con el mismo horizonte y atraso
  (*H* = 8, *X* = 14), *K* = 4 semanas alcanza un AUC de validación de 0,831 frente a 0,802
  con *K* = 8 y 0,799 con *K* = 12. Esperar más semanas no mejora la predicción en validación
  y retrasa la alerta. La población no es la misma (con *K* = 4 hay más episodios porque
  entran los que se cierran antes de la semana 8), por lo que se interpreta como una
  tendencia y no como una comparación exacta.
- **Cuanto más lejos el horizonte, peor.** Con *H* = 16 el AUC de validación suele ser menor
  que con *H* = 8 en la misma *K* y *X*.
- **La prueba con *K* = 12 es engañosa.** Su AUC de prueba es el más alto (hasta 0,85) pero
  la población de prueba es otra (solo episodios que sobreviven 12 semanas) y es pequeña
  (≈ 500 episodios); el intervalo de confianza del modelo base en prueba (sección 6) mide
  cerca de ± 0,04, así que diferencias de 0,03 o menos no son concluyentes.

Para la definición del estudio se mantiene *K* = 8, *H* = 12 y *X* = 14, fijada antes de
explorar. Los resultados de esta sección sugieren evaluar con el equipo de negocio si una
alerta a las 4 semanas, que es más temprana, es operativamente más valiosa.

## 6. Modelo final y evaluación en prueba

Se toma el modelo con mayor AUC de validación (sección 2) y se evalúa **una sola vez** en
la prueba temporal, con el umbral elegido en validación. Se acompaña de un intervalo de
confianza del 95 % por *bootstrap* sobre los episodios de prueba.

In [ ]:
r = resultados[MEJOR]
y_te, p_te, u = r["y_prueba"], r["p_prueba"], r["umbral"]
rng = np.random.default_rng(SEMILLA)
from sklearn.metrics import roc_auc_score
boot = []
for _ in range(1000):
    idx = rng.integers(0, len(y_te), len(y_te))
    if len(np.unique(y_te[idx])) > 1:
        boot.append(roc_auc_score(y_te[idx], p_te[idx]))
ic = np.percentile(boot, [2.5, 97.5])
print(f"{MEJOR}: AUC en prueba = {r['prueba']['AUC']:.3f}  (IC 95 %: {ic[0]:.3f} – {ic[1]:.3f})")
print({k: round(v, 3) for k, v in r["prueba"].items()}, "| umbral =", round(u, 2))
CIFRAS["modelo_final"] = {"modelo": MEJOR, "umbral": u, "prueba": r["prueba"], "auc_ic95": ic.tolist()}

fig, ax = plt.subplots(1, 2, figsize=(9, 3.8))
fpr, tpr, _ = roc_curve(y_te, p_te)
ax[0].plot(fpr, tpr, color=AZUL, label=f"AUC = {r['prueba']['AUC']:.3f}")
ax[0].plot([0, 1], [0, 1], color=TINTA_2, ls="--", lw=0.8)
ax[0].set_xlabel("1 − especificidad"); ax[0].set_ylabel("sensibilidad"); ax[0].set_title("Curva ROC (prueba)")
ax[0].legend(frameon=False)
cm = confusion_matrix(y_te, (p_te >= u).astype(int))
ax[1].imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm):
    ax[1].text(j, i, v, ha="center", va="center", color="white" if v > cm.max() / 2 else TINTA, fontsize=12)
ax[1].set_xticks([0, 1]); ax[1].set_yticks([0, 1]); ax[1].set_xticklabels(["sin riesgo", "riesgo grave"])
ax[1].set_yticklabels(["sin riesgo", "riesgo grave"]); ax[1].grid(False)
ax[1].set_xlabel("predicho"); ax[1].set_ylabel("real"); ax[1].set_title("Matriz de confusión (prueba)")
fig.tight_layout(); fig.savefig(DIR_FIG / "modelo_final.png"); plt.show()

**Lectura.** El modelo base logra un AUC de aproximadamente 0,76 en el periodo más reciente
(intervalo de confianza del 95 %: 0,72 a 0,79), con una exactitud balanceada de 0,69 y un F1
de 0,62 para la clase de riesgo grave. Es un desempeño moderado pero útil como filtro de
priorización de cobro, y es el que se obtiene al validar en las condiciones del uso real
(predecir el futuro con datos del pasado). El AUC cercano a 0,81 de la prueba rápida con
partición aleatoria era optimista porque mezclaba periodos.

In [ ]:
(DIR_SALIDA / "reports" / "cifras_clasificacion.json").write_text(
    json.dumps(CIFRAS, indent=2, ensure_ascii=False, default=lambda o: float(o) if hasattr(o, "item") else str(o)),
    encoding="utf-8")
print("Resultados guardados en", DIR_SALIDA / "reports")

# PARTE 4 · Selección de variables y explicabilidad

<sub>Contenido del cuaderno `04_seleccion_explicabilidad.ipynb`.</sub>

# Entregable II — Selección de variables, reducción de dimensión y explicabilidad

Modelos y Simulación de Sistemas II — Universidad de Antioquia — Grupo 14

Este notebook responde tres preguntas sobre el clasificador de riesgo grave de mora:

1. ¿Qué variables tienen más relación con el riesgo, vistas una por una (análisis de
   filtro)?
2. ¿Se puede **simplificar el modelo** (menos variables o menos dimensiones) sin perder
   desempeño? Se comparan selección por información mutua, regresión logística con
   penalización L1 (Lasso), PCA y UMAP.
3. ¿**Qué explica las predicciones**? Coeficientes de la regresión logística, importancia
   por permutación y valores SHAP del Random Forest.

**Mismo protocolo que el notebook 03.** Partición temporal (el 20 % más reciente es la
prueba), validación con pliegues de ventana expansiva, y todo paso que aprende de los datos
(imputación, escalado, selección, PCA) dentro del `Pipeline`, reajustado en cada pliegue.
El análisis individual de la sección 2 usa **solo los episodios de desarrollo**.

**Configuración base.** *K* = 8, *H* = 12, *X* = 14. Según la sección 3 del notebook 03, ni el
perfil de la etapa 1 ni las marcas de gestión mejoran el modelo, así que aquí se trabaja sin
ellos: es el modelo más simple y no depende de decisiones del cobrador.

In [ ]:
#@title ⚙️ Preparación de esta parte (no hace falta tocar)
import functools
import json
import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
EN_COLAB = "google.colab" in sys.modules
NOMBRE_CARPETA = "ProyectoModelos2_Colab"


def buscar_carpeta_drive():
    raices = [Path("/content/drive/MyDrive"), *Path("/content/drive").glob(".shortcut-targets-by-id/*")]
    for patron in (NOMBRE_CARPETA, f"*/{NOMBRE_CARPETA}", f"*/*/{NOMBRE_CARPETA}"):
        for raiz in raices:
            for carpeta in raiz.glob(patron):
                if (carpeta / "clasificacion.py").exists():
                    return carpeta
    return None


if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    carpeta = buscar_carpeta_drive()
    if carpeta is None:
        raise FileNotFoundError(
            f"No se encontro la carpeta {NOMBRE_CARPETA} en su Drive. Agreguele un acceso directo "
            "en Mi unidad (Compartidos conmigo > clic derecho > Organizar > Agregar acceso directo) "
            "y vuelva a ejecutar todo.")
    os.chdir(carpeta)
    get_ipython().system("pip install -q shap umap-learn")
RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DIR_SALIDA = Path("/content/salidas") if EN_COLAB else RAIZ


def ubicar(archivos, carpetas):
    for c in carpetas:
        if all((c / a).exists() for a in archivos):
            return c
    raise FileNotFoundError(f"No se encontraron {archivos} en {[str(c) for c in carpetas]}.")


sys.path.insert(0, str(ubicar(["clasificacion.py", "ciclos.py"], [RAIZ / "src", Path.cwd()])))
DIR_DATOS = ubicar(["abonos.csv.gz", "motos_diario.csv.gz"], [RAIZ / "data" / "anon", Path.cwd()])
DIR_FIG = DIR_SALIDA / "reports" / "figuras"
DIR_FIG.mkdir(parents=True, exist_ok=True)

import shap
import umap
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel, SelectKBest, mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline

from clasificacion import CAT, MARCAS, NUM_BASE, Preprocesador, cargar_base, construir_dataset, evaluar
from particion import corte_por_fraccion, dividir_temporal, pliegues_expansivos

TINTA, TINTA_2, REJILLA = "#0b0b0b", "#52514e", "#e6e5e0"
AZUL, NARANJA, VERDE = "#2a78d6", "#eb6834", "#2a9d6f"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 10,
    "axes.edgecolor": TINTA_2, "axes.labelcolor": TINTA, "xtick.color": TINTA_2, "ytick.color": TINTA_2,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": REJILLA,
    "axes.axisbelow": True, "axes.titlesize": 11, "axes.titleweight": "bold",
})
CIFRAS = {}
SEMILLA = 0

## 1. Datos y partición

In [ ]:
ep, ciclos, pagos, diario, CORTE = cargar_base(DIR_DATOS)
K, H, X = 8, 12, 14
data = construir_dataset(ep, ciclos, pagos, diario, CORTE, pd.Timestamp("2024-03-30"), K, H, X)
COLS = NUM_BASE + CAT                       # sin marcas de gestión
corte = corte_por_fraccion(data.fecha_inicio, 0.2)
i_dev, i_te = dividir_temporal(data.fecha_inicio, K, H, X, corte)
dev, te = data.iloc[i_dev].reset_index(drop=True), data.iloc[i_te].reset_index(drop=True)
print(f"Desarrollo: {len(dev)} episodios ({100 * dev.y.mean():.1f} % positivos) | prueba: {len(te)} ({100 * te.y.mean():.1f} %)")

def base(extra=(), clf=None):
    clf = clf if clf is not None else LogisticRegression(C=0.5, max_iter=2000, class_weight="balanced")
    return Pipeline([("pre", Preprocesador(con_perfil=False, con_marcas=False, seed=SEMILLA)), *extra, ("clf", clf)])

## 2. Análisis individual de las variables (filtro)

Con los episodios de **desarrollo** se mide, para cada variable ya imputada y estandarizada,
tres señales de su relación con el riesgo grave: el **AUC univariado** (qué tan bien ordena
los episodios por sí sola; se reporta el valor mayor entre AUC y 1 − AUC para ignorar el
signo), la **información mutua** con la etiqueta y la **correlación de Spearman**. Los
indicadores `missingindicator_*` marcan si el valor original faltaba (por ejemplo, un
episodio sin pagos en la ventana).

In [ ]:
pre = Preprocesador(con_perfil=False, con_marcas=False, seed=SEMILLA).fit(dev[COLS])
Z = pre.transform(dev[COLS])
nombres = pre.get_feature_names_out()
y_dev = dev.y.to_numpy()
mi = mutual_info_classif(Z, y_dev, random_state=SEMILLA)
filas = []
for j, n in enumerate(nombres):
    if Z[:, j].std() == 0:
        continue
    auc = roc_auc_score(y_dev, Z[:, j])
    filas.append({"variable": n, "AUC univariado": max(auc, 1 - auc), "signo": "+" if auc >= 0.5 else "−",
                  "información mutua": mi[j], "Spearman": spearmanr(Z[:, j], y_dev)[0]})
indiv = pd.DataFrame(filas).set_index("variable").sort_values("AUC univariado", ascending=False)
display(indiv.head(15).round(3))
CIFRAS["individual_top15"] = indiv.head(15).round(4).to_dict(orient="index")

top = indiv.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.barh(top.index, top["AUC univariado"], color=AZUL)
ax.axvline(0.5, color=TINTA_2, lw=0.8, ls="--")
ax.set_xlim(0.45, top["AUC univariado"].max() + 0.03)
ax.set_xlabel("AUC univariado (desarrollo)"); ax.set_title("Variables más asociadas al riesgo grave")
fig.tight_layout(); fig.savefig(DIR_FIG / "auc_univariado.png"); plt.show()

num_dev = pd.DataFrame(Z, columns=nombres)[[n for n in nombres if n in NUM_BASE]]
corr = num_dev.corr(method="spearman").abs()
pares = (corr.where(np.triu(np.ones(corr.shape, bool), 1)).stack().sort_values(ascending=False))
pares = pares[pares > 0.8]
print("Pares de variables con |Spearman| > 0,8 (redundantes):")
display(pares.round(2).to_frame("|ρ|"))
CIFRAS["pares_redundantes"] = {f"{a} ~ {b}": round(float(v), 3) for (a, b), v in pares.items()}

**Lectura.** Las variables más asociadas al riesgo son las de atraso: el atraso promedio
(AUC univariado 0,759), el déficit relativo de pago (0,749), la proporción de ciclos con atraso
(0,731) y, en sentido contrario, el pago total al canon (0,727: pagar más protege). Ninguna
supera por sí sola 0,76, así que el modelo se apoya en varias variables a la vez. La empresa
también se asocia (RENTAYA, 0,646). Hay cuatro pares de variables casi duplicadas (|ρ| > 0,8),
lo que confirma las redundancias que ya se habían visto en el EDA; por ejemplo, el atraso
promedio y la proporción de ciclos atrasados (0,97).

## 3. Selección y reducción de variables dentro de la validación

Se comparan, con la misma validación temporal, la regresión logística y el Random Forest
usando: todas las variables; las *k* mejores por información mutua (*k* = 5, 8, 12);
la selección por **Lasso** (regresión logística con penalización L1, que lleva a cero los
coeficientes poco útiles); **PCA** con 5, 8 y 12 componentes; y **UMAP** de 5 dimensiones.
La selección y la proyección se **reajustan en cada pliegue** con sus datos de
entrenamiento. UMAP no usa la etiqueta, así que aquí sirve de prueba de qué tanto sirve una
reducción no supervisada de la estructura local como entrada de un clasificador.

In [ ]:
mi_fn = functools.partial(mutual_info_classif, random_state=SEMILLA)
lasso = LogisticRegression(penalty="l1", solver="liblinear", C=0.1, class_weight="balanced")
RF = lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=10, class_weight="balanced", n_jobs=-1, random_state=SEMILLA)
variantes = {
    "todas las variables": [],
    "información mutua, k = 5": [("sel", SelectKBest(mi_fn, k=5))],
    "información mutua, k = 8": [("sel", SelectKBest(mi_fn, k=8))],
    "información mutua, k = 12": [("sel", SelectKBest(mi_fn, k=12))],
    "Lasso (L1)": [("sel", SelectFromModel(lasso))],
    "PCA, 5 componentes": [("pca", PCA(5, random_state=SEMILLA))],
    "PCA, 8 componentes": [("pca", PCA(8, random_state=SEMILLA))],
    "PCA, 12 componentes": [("pca", PCA(12, random_state=SEMILLA))],
    "UMAP, 5 dimensiones": [("umap", umap.UMAP(n_components=5, random_state=SEMILLA))],
}
filas = []
for nombre, extra in variantes.items():
    for modelo, mk in (("Regresión logística", None), ("Random Forest", RF)):
        if nombre.startswith("UMAP") and modelo == "Random Forest":
            continue
        r = evaluar(base(extra, mk() if mk else None), data, COLS, K, H, X)
        filas.append({"reducción": nombre, "modelo": modelo,
                      "AUC validación": f"{r['auc_validacion']:.3f} ± {r['auc_validacion_sd']:.3f}",
                      "AUC prueba": round(r["prueba"]["AUC"], 3), "BACC prueba": round(r["prueba"]["BACC"], 3),
                      "MCC prueba": round(r["prueba"]["MCC"], 3)})
reduccion = pd.DataFrame(filas)
display(reduccion)
CIFRAS["reduccion"] = reduccion.to_dict(orient="records")

**Lectura.** Simplificar el modelo no hace perder desempeño de forma clara. Con solo 5
variables elegidas por información mutua, la regresión logística tiene un AUC de validación de
0,788 frente a 0,809 con todas (una diferencia menor que la variabilidad entre pliegues, ≈ 0,03)
y en prueba 0,773 frente a 0,757; el Lasso conserva 0,800 y el Random Forest 0,802. PCA no
aporta ventaja (0,77 a 0,80 según el modelo y el número de componentes) y UMAP como entrada del
clasificador rinde mucho peor (0,665), lo esperable de una proyección que no usa la etiqueta.
Las diferencias entre validación y prueba van en sentidos opuestos, así que no son
concluyentes. Lo útil es que un modelo de 5 a 8 variables rinde de forma equivalente a uno con
todas y es más fácil de explicar y de calcular en la empresa.

### 3.1 ¿Qué variables eligen los métodos y qué tan estable es la elección?

Se cuenta en cuántos de los pliegues de validación cada variable es elegida por la
información mutua (*k* = 8) y por el Lasso. Una variable elegida en todos los pliegues es una
señal estable; una elegida en uno solo depende del periodo.

In [ ]:
pliegues = [(tr, va) for tr, va in pliegues_expansivos(data.fecha_inicio, K, H, X, 4, fecha_max=corte) if len(tr) >= 100]
conteo = {"información mutua (k = 8)": {}, "Lasso": {}}
for tr, _ in pliegues:
    d = data.iloc[tr]
    for etiqueta_met, sel in (("información mutua (k = 8)", SelectKBest(mi_fn, k=8)), ("Lasso", SelectFromModel(lasso))):
        p = Pipeline([("pre", Preprocesador(False, False, seed=SEMILLA)), ("sel", sel)]).fit(d[COLS], d.y)
        elegidas = p.named_steps["pre"].get_feature_names_out()[p.named_steps["sel"].get_support()]
        for n in elegidas:
            conteo[etiqueta_met][n] = conteo[etiqueta_met].get(n, 0) + 1
est = pd.DataFrame(conteo).fillna(0).astype(int)
est = est.assign(total=est.sum(axis=1)).sort_values("total", ascending=False).drop(columns="total")
est.index.name = f"variable (de {len(pliegues)} pliegues)"
display(est.head(15))
CIFRAS["estabilidad_seleccion"] = est.head(15).to_dict(orient="index")

**Lectura.** Tanto la información mutua como el Lasso eligen en todos los pliegues la
proporción de ciclos con 3 o más días de atraso, los ciclos sin pagar, la variabilidad del pago
(`cv_pago_canon`) y el déficit relativo. La información mutua además conserva las variables de
atraso y pago redundantes entre sí; el Lasso las descarta y se queda con la multa acumulada, las
justificaciones por inmovilización y la empresa. Es una señal estable: el núcleo de variables
relevantes no depende del periodo usado para elegirlas.

## 4. PCA: cuánta información comprimen las componentes

In [ ]:
pca = PCA(random_state=SEMILLA).fit(Z)
acum = np.cumsum(pca.explained_variance_ratio_)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].bar(range(1, 13), pca.explained_variance_ratio_[:12], color=AZUL)
ax[0].plot(range(1, 13), acum[:12], color=NARANJA, marker="o")
ax[0].set_xlabel("componente"); ax[0].set_ylabel("varianza explicada"); ax[0].set_title("Varianza explicada y acumulada")
cargas = pd.DataFrame(pca.components_[:2].T, index=nombres, columns=["PC1", "PC2"])
c1 = cargas.reindex(cargas.PC1.abs().sort_values(ascending=False).index).head(8).iloc[::-1]
ax[1].barh(c1.index, c1.PC1, color=AZUL); ax[1].set_title("Mayores cargas de la 1.ª componente")
fig.tight_layout(); fig.savefig(DIR_FIG / "pca_varianza.png"); plt.show()
n90 = int(np.searchsorted(acum, 0.9) + 1)
print(f"Se necesitan {n90} componentes para explicar el 90 % de la varianza ({len(nombres)} variables).")
CIFRAS["pca"] = {"n_variables": len(nombres), "componentes_90": n90, "varianza_pc1_pc2": pca.explained_variance_ratio_[:2].round(4).tolist()}

**Lectura.** La información está repartida: la primera componente explica solo el 39 % de la
varianza, las dos primeras el 52 %, y se necesitan 10 componentes de 28 variables para llegar al
90 %. No hay una dimensión dominante, lo que explica que PCA no mejore la clasificación.

## 5. Explicabilidad

Se ajusta cada modelo con todo el desarrollo y se explica su comportamiento. Los resultados
describen **qué usa el modelo**, no causas: que una variable se asocie con el riesgo no
significa que provocarlo.

### 5.1 Regresión logística: coeficientes

Con variables estandarizadas, el coeficiente indica cuánto cambia el logaritmo de las
probabilidades de riesgo grave por cada desviación estándar de la variable.

In [ ]:
logit = LogisticRegression(C=0.5, max_iter=2000, class_weight="balanced").fit(Z, y_dev)
coef = pd.Series(logit.coef_[0], index=nombres)
coef = coef[coef.abs().sort_values(ascending=False).index]
fig, ax = plt.subplots(figsize=(6.5, 4.2))
c = coef.head(12).iloc[::-1]
ax.barh(c.index, c.values, color=[NARANJA if v > 0 else AZUL for v in c.values])
ax.axvline(0, color=TINTA_2, lw=0.8)
ax.set_xlabel("coeficiente (variable estandarizada)"); ax.set_title("Regresión logística: mayores coeficientes")
fig.tight_layout(); fig.savefig(DIR_FIG / "coeficientes_logistica.png"); plt.show()
CIFRAS["coeficientes_top12"] = coef.head(12).round(4).to_dict()

**Lectura.** Los mayores coeficientes son el atraso promedio (+0,96), la multa acumulada
(+0,52) y los ciclos sin pagar (+0,50): más atraso, más multas y más cuotas sin pagar
aumentan el riesgo. La empresa aparece con coeficientes grandes y opuestos
(RENTAYA +0,99, MOTOHOY −0,93); son dos columnas complementarias de una misma variable, así
que lo que importa es su diferencia, no cada valor por separado.

### 5.2 Random Forest: importancia por permutación y valores SHAP

La **importancia por permutación** mide cuánto cae el AUC cuando se baraja una variable
(se calcula sobre la prueba, solo para describir el modelo final). Los **valores SHAP**
reparten la predicción de cada episodio entre las variables; el gráfico muestra, para cada
variable, cómo se reparte su efecto según sea alto o bajo su valor.

In [ ]:
rf = RF().fit(Z, y_dev)
Z_te = pre.transform(te[COLS])
perm = permutation_importance(rf, Z_te, te.y, scoring="roc_auc", n_repeats=20, random_state=SEMILLA)
imp = pd.Series(perm.importances_mean, index=nombres).sort_values(ascending=False)
display(imp.head(10).round(4).to_frame("caída del AUC al barajar"))
CIFRAS["permutacion_top10"] = imp.head(10).round(4).to_dict()

expl = shap.TreeExplainer(rf)
sv = expl(Z_te)
valores = sv.values[:, :, 1] if sv.values.ndim == 3 else sv.values
shap.summary_plot(valores, Z_te, feature_names=nombres, max_display=12, show=False)
plt.title("Valores SHAP del Random Forest (clase: riesgo grave)")
plt.tight_layout(); plt.savefig(DIR_FIG / "shap_resumen.png"); plt.show()
shap_medio = pd.Series(np.abs(valores).mean(axis=0), index=nombres).sort_values(ascending=False)
CIFRAS["shap_top10"] = shap_medio.head(10).round(4).to_dict()

**Lectura.** Según la permutación, las variables que más pesan son el déficit relativo (el
AUC cae 0,026 al barajarla) y el atraso promedio (0,021). Según SHAP, las que más contribuyen
son la empresa (RENTAYA), el déficit relativo y el atraso promedio. En el gráfico SHAP los
valores altos de atraso y de déficit empujan la predicción hacia riesgo grave, y los valores altos de pago
total la empujan hacia lo contrario.

### 5.3 ¿Coinciden los métodos?

Si métodos muy distintos (un modelo lineal, un bosque y un filtro) señalan las mismas
variables, la conclusión es más confiable.

In [ ]:
rankings = pd.DataFrame({
    "AUC univariado": indiv["AUC univariado"],
    "|coef.| logística": coef.abs(),
    "permutación RF": imp,
    "|SHAP| RF": shap_medio,
}).dropna()
rho = rankings.rank().corr(method="spearman").round(2)
display(rho)
top5 = {c: list(rankings[c].sort_values(ascending=False).head(5).index) for c in rankings.columns}
for c, v in top5.items():
    print(f"{c}: {', '.join(v)}")
CIFRAS["acuerdo_rankings"] = rho.to_dict()
CIFRAS["top5_por_metodo"] = top5

**Lectura.** Los métodos coinciden en lo esencial: el atraso promedio, el déficit relativo,
la proporción de ciclos con 3 o más días de atraso y los ciclos sin pagar aparecen entre las
cinco variables principales en varios de ellos. El AUC univariado y SHAP concuerdan bastante
(ρ = 0,87); la regresión logística y la permutación concuerdan menos (0,34), porque entre
variables casi duplicadas los modelos reparten la importancia de forma distinta.

**Dos cautelas.** (1) Estas importancias describen qué usa el modelo, no causas. (2) La
**empresa** tiene un peso grande en la regresión logística y en SHAP. Es una característica
del contrato conocida desde el inicio, pero conviene confirmar con la empresa que el campo es
confiable y qué diferencia a RENTAYA de MOTOHOY (por ejemplo, el tipo de cliente o de moto),
porque un efecto tan grande puede reflejar una diferencia de operación y no de comportamiento.

In [ ]:
(DIR_SALIDA / "reports" / "cifras_seleccion.json").write_text(
    json.dumps(CIFRAS, indent=2, ensure_ascii=False, default=lambda o: float(o) if hasattr(o, "item") else str(o)),
    encoding="utf-8")
print("Resultados guardados en", DIR_SALIDA / "reports")

# PARTE 5 · Búsqueda de un mejor modelo y validación externa

<sub>Contenido del cuaderno `05_mejora_modelo.ipynb`.</sub>

# Entregable II — Búsqueda de un mejor modelo y validación externa

Modelos y Simulación de Sistemas II — Universidad de Antioquia — Grupo 14

Con el modelo base del notebook 03 (AUC de validación ≈ 0,80 y de prueba ≈ 0,76 a 0,79) se
probaron cuatro maneras de mejorarlo, todas con **el mismo protocolo temporal** (K = 8, H = 12,
X = 14; validación de ventana expansiva; prueba usada una sola vez). Después se evaluó el modelo
de Cartagena en otras dos sedes. Los experimentos se corren desde `experimentos/` y este
cuaderno solo lee sus resultados (`reports/`).

1. Variables de **trayectoria** (cómo termina la ventana de observación).
2. **Ajuste de hiperparámetros** y **ensamble**.
3. Información **nueva de la base**: gestión de cobranza y justificaciones (solo eventos con fecha ≤ semana K).
4. **Validación externa** en Sincelejo y Montería.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
R = Path("../reports") if Path("../reports").exists() else Path("reports")
pd.options.display.float_format = "{:.3f}".format

## 1. Variables de trayectoria

Últimas dos semanas de la ventana, tendencia y desviación del atraso, cambio entre mitades, días desde el último pago, mayor hueco entre pagos y dinero pagado en las últimas dos semanas.

In [ ]:
t = pd.read_csv(R / "mejora_trayectoria.csv")
display(t)

Las diferencias (|ΔAUC| ≤ 0,01 en validación en los modelos principales) quedan dentro de la desviación entre pliegues (≈ 0,02): la trayectoria no aporta información que las variables base no tengan.

## 2. Ajuste de hiperparámetros y ensamble

La configuración se elige solo con el AUC de validación temporal; la prueba se reporta pero no decide.

In [ ]:
a = pd.read_csv(R / "ajuste_hiperparametros.csv")
a["familia"] = a.modelo.str.split().str[0]
display(a.sort_values("AUC val", ascending=False).groupby("familia").head(3).sort_values(["familia", "AUC val"], ascending=[True, False]))

In [ ]:
e = pd.read_csv(R / "mejora_ensamble.csv")
display(e)

El mejor ajuste mejora el AUC de validación en menos de 0,01 y el ensamble (voto suave de logística, Random Forest y Gradient Boosting) queda igual que Random Forest solo. La precisión en el top-k es la métrica de uso práctico: si la empresa revisara solo el 10 % de episodios con mayor riesgo, acertaría cerca de 8 de cada 10 (tasa base ≈ 39 %).

## 3. Información nueva de la base

`np_justi_super` (justificaciones) y `np_gestion` (gestión de cobranza), solo lectura y anonimizadas. Se cuentan los eventos con fecha entre el inicio y la semana K; el estado aprobado/rechazado se ignora porque la empresa lo actualiza después (fuga). `tests/test_ciclos.py` verifica que eventos posteriores a K no cambian las variables.

In [ ]:
g = pd.read_csv(R / "mejora_gestion.csv")
display(g)

Las variables se asocian con la etiqueta (p. ej. más registros de justificación y cuotas dobles en los episodios de riesgo), pero repiten lo que ya dicen los pagos: no mejoran la validación. Además `np_gestion` casi no tiene registros en 2024.

## 4. Validación externa: entrenar en Cartagena, probar en otras sedes

Sincelejo y Montería usan la misma estructura de base en otros servidores. El modelo se entrena con todos los episodios de Cartagena y predice, sin reajustarse, los de cada sede (cohorte desde su primer pago registrado).

In [ ]:
s = pd.read_csv(R / "validacion_externa_sedes.csv")
display(s[s.vars == "con empresa"].drop(columns="vars"))
fig, ax = plt.subplots(figsize=(6, 3.2))
p = s[s.vars == "con empresa"].pivot(index="modelo", columns="sede", values="AUC")
p.plot.bar(ax=ax, rot=0); ax.set_ylim(0.6, 0.85); ax.set_ylabel("AUC"); ax.set_title("AUC por sede con el modelo entrenado en Cartagena")
plt.tight_layout(); plt.show()

El AUC en las otras sedes (0,75 a 0,81) es igual o mayor que el de la prueba temporal de Cartagena, aunque su tasa de riesgo es menor (21 a 24 % frente a 36 %). Quitar `empresa` apenas cambia el resultado (±0,003): en las sedes nuevas todos los episodios son MOTOHOY. Montería tiene solo 199 episodios, por lo que su cifra es menos precisa; la conclusión se apoya sobre todo en Sincelejo.

## Conclusión

Ninguna de las cuatro vías de mejora aumenta el AUC más allá del ruido de la validación: con la información de las primeras K semanas el modelo ya está cerca de su máximo. El resultado más sólido es que **el modelo generaliza a otras sedes sin reajustarse**. El umbral de decisión, en cambio, debería calibrarse por sede porque la tasa de riesgo cambia.

# Control de calidad final

In [ ]:
#@title ✅ Control de calidad final: ¿los resultados coinciden con el informe?
import json
from pathlib import Path

R = Path("/content/salidas/reports") if "google.colab" in __import__("sys").modules else Path("reports")


def leer(nombre):
    return json.loads((R / nombre).read_text(encoding="utf-8"))


resultados = []


def revisar(nombre, obtenido, esperado, tol):
    ok = abs(obtenido - esperado) <= tol
    resultados.append(ok)
    print(("  ✅ " if ok else "  ❌ ") + f"{nombre}: {obtenido:.3f} (esperado {esperado:.3f} ± {tol})")


eda, clu, cla, sel = (leer(f"cifras_{n}.json") for n in ("eda", "clustering", "clasificacion", "seleccion"))
print("Población y etiqueta")
revisar("episodios del modelo", cla["datos"]["n"], 2889, 0)
revisar("proporción de riesgo grave", cla["datos"]["pos"], 0.355, 0.002)
revisar("episodios en desarrollo", cla["datos"]["n_dev"], 1735, 0)
revisar("episodios en prueba", cla["datos"]["n_prueba"], 582, 0)
print("Clustering")
revisar("episodios entrenamiento (partición)", clu["particion"]["train"], 1735, 0)
print("Clasificación (AUC de prueba)")
for modelo, esperado in (("Regresión logística", 0.756), ("Random Forest", 0.784), ("Gradient Boosting", 0.783)):
    revisar(modelo, cla["modelos"][modelo]["AUC (prueba)"], esperado, 0.02)
print("Rejilla K/H/X")
revisar("combinaciones evaluadas", len(cla["rejilla"]), len(cla["rejilla"]), 0)
n_ok = sum(resultados)
print(f"\n{n_ok} de {len(resultados)} controles correctos.")
print("🎉 Todo el cuaderno corre y reproduce el informe." if n_ok == len(resultados)
      else "⚠️ Hay diferencias: revise las líneas con ❌ (pequeñas variaciones por versión de librerías son normales).")